# Domain II: Chess - Stage II Emergence Validation

This notebook implements the chess validation domain (§5.3, §7.2) of the Informational Buildup Framework (IBF). Chess tests whether the mechanisms confirmed in the controlled RRW domain (Stage I) produce **emergent behavior under genuine strategic complexity**, evaluated by an external oracle (Stockfish) rather than by a built-in analytical score.

**Three predictions are tested (§6.3):**

- **P4. Behavioral Advantage**: The fully adapted IBF agent achieves a statistically significant centipawn advantage over the passive baseline under independent Stockfish evaluation. The value-only ablation confirms the *argmax identity* at inference time; the No-Agency training ablation isolates the developmental contribution of agency.
- **P5.Emergent Spatial Agency**: The agent autonomously develops spatially nonuniform k_eff(z) correlated with positional structure, not merely with legal-move count.
- **P6.Transfer and Retention Under Crucible Verification**: Verified corrections carry knowledge across context boundaries, producing positive backward transfer that exceeds the replay baseline without storing raw data.

**Domain setup:**
- Latent space: 12D (8D frozen board encoding from a 5-head CNN + 4D move features)
- Three sequential contexts: A (materially imbalanced), B (quiet/balanced), C (restricted mobility)
- Training oracle: Stockfish depth 4 on disjoint positions
- Evaluation oracle: Stockfish depth 8, deterministic, n = 1,002 positions, ±1000 cp symmetric clip
- Geometric calibration: σ* derived from latent geometry (no grid search)
- Seed 42 (primary run); seeds 43–44 (supplementary replication, C11)

In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C1: SETUP & DEPENDENCIES                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os, gc, copy, time, random, shutil, warnings, pickle
import json as json_mod

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0: print(f"WARN: {cmd}")
    return r.returncode == 0

_run("pip install -q python-chess zstandard scipy faiss-cpu")

import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
import chess, chess.engine, chess.pgn
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda': print(f"  GPU: {torch.cuda.get_device_name(0)}")

SF_PATH = shutil.which("stockfish") or "/usr/local/bin/stockfish"
assert os.path.exists(SF_PATH), "Stockfish not found"

# One-shot verification probe — opens, checks, quits immediately.
# BUG 5 FIX: no persistent engine left open from C1.
_probe = chess.engine.SimpleEngine.popen_uci(SF_PATH)
_info  = _probe.analyse(chess.Board(), chess.engine.Limit(depth=5))
print(f"Stockfish OK: {_info['score'].relative.score(mate_score=10000)} cp")
_probe.quit()
del _probe, _info

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print("\n✓ C1 complete")

Device: cuda
  GPU: NVIDIA GeForce RTX 5090
Stockfish OK: 50 cp

✓ C1 complete


In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C2: CONFIGURATION                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
CHECKPOINT_DIR      = '/workspace/ibf_chess_paper/'
PGN_PATH            = '/workspace/lichess_elite_2024-01.pgn'
DEV_ENCODER_PATH    = '/workspace/ibf_8x8/model_C_v2_GO_backup.pt'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Data pipeline ──────────────────────────────────────────────────────────────
N_POSITIONS         = 200_000
MAX_GAMES_TO_PARSE  = 50_000
POSITIONS_PER_GAME  = 3
MOVE_RANGE          = (10, 40)

# ── Encoder ────────────────────────────────────────────────────────────────────
ENCODER_Z_DIM       = 8
ENCODER_SF_DEPTH    = 8
ENCODER_N_POSITIONS = 15000
ENCODER_EPOCHS      = 60
ENCODER_PATIENCE    = 15
ENCODER_LR          = 1e-3
ENCODER_DROPOUT     = 0.3
ENCODER_LAMBDA_DECORR = 1.0

# ── σ / geometry — live values set in C5 from calibration geometry ─────────────
# SIGMA_TRAIN, SIGMA_FLOOR, MERGE_THRESHOLD, SIGMA_AGENCY, kappa_chess
# are NOT set here; they are computed in C5 from P10/3 and sib/20.
# Do not hardcode them here. C8 reads from globals set by C5.

# ── IBF dynamics ───────────────────────────────────────────────────────────────
V_MAX                    = 0.50
W_MAX                    = 5.0
ETA                      = 0.1
ETA_CRYST                = 0.005
ETA_K                    = 0.10
MU_BASE                  = 0.06
MU_CRYSTALLIZED          = 0.001
CONVERGENCE_THRESHOLD    = 0.025
CRYSTALLIZATION_THRESHOLD = 15
N_CROSS_MIN              = 8
REVERSAL_THRESHOLD       = -0.0085
W_DVAR_THRESHOLD         = 0.005
ALPHA_SHRINK             = 500.0
CAPACITY                 = 50000

# ── Training protocol ──────────────────────────────────────────────────────────
EPOCHS_PER_PHASE    = 120
GAMES_PER_EPOCH     = 100
MAX_MOVES_PER_GAME  = 30
N_EVAL_POSITIONS    = 1000
N_EVAL_TRAINING     = 500
K_0                 = 5.0
MOVE_SCALE          = 10.0
SF_DEPTH_TRAIN      = 4
SF_DEPTH_EVAL       = 8
TRAIN_PER_CONTEXT   = 1000
EVAL_PER_CONTEXT    = 1000

# ── Stockfish factory ──────────────────────────────────────────────────────────
# No persistent global engines. Every cell that needs Stockfish
# calls get_sf_engine() and is responsible for closing it.
def get_sf_engine(threads=1, hash_mb=128):
    eng = chess.engine.SimpleEngine.popen_uci(SF_PATH)
    eng.configure({"Threads": threads, "Hash": hash_mb})
    return eng

# ── Stockfish eval: RAW (training path) ───────────────────────────────────────
# Returns raw centipawn score. Mate scores return ±10000.
# NO CLIPPING. This function is used ONLY for computing D-signals
# during training (compute_D_and_update via R_imposed_override).


def stockfish_eval_cp_raw(board, sf_eng, depth=SF_DEPTH_EVAL):
    try:
        info  = sf_eng.analyse(board, chess.engine.Limit(depth=depth))
        score = info["score"].relative
        if score.is_mate():
            return 10000 if score.mate() > 0 else -10000
        else:
            return score.score()
    except Exception as e:
        print(f"  WARNING: SF eval failed: {e}")
        return 0

# ── Stockfish eval: CLIPPED (evaluation path) ─────────────────────────────────
# Returns centipawn score clipped at ±CP_CLIP_EVAL.
# Mate scores (±10000) are categorical outcomes, not positional evaluations.
# Mixing them with centipawn scores corrupts means.
# This function is used for ALL evaluation metrics: passive baseline,
# phase-end eval, C8b ablation eval, C9 final eval, C10 σ sweep.
#
# The clip is symmetric and applied ONLY here, never in the training path.

CP_CLIP_EVAL = 1000

def stockfish_eval_cp_clipped(board, sf_eng, depth=SF_DEPTH_EVAL):
    raw = stockfish_eval_cp_raw(board, sf_eng, depth)
    return max(-CP_CLIP_EVAL, min(CP_CLIP_EVAL, raw))

print(f"✓ C2 loaded. PGN: {'found' if os.path.exists(PGN_PATH) else 'NOT FOUND'}")
print(f"  Training SF eval: stockfish_eval_cp_raw (unclipped)")
print(f"  Evaluation SF eval: stockfish_eval_cp_clipped (±{CP_CLIP_EVAL} cp)")

✓ C2 loaded. PGN: found
  Training SF eval: stockfish_eval_cp_raw (unclipped)
  Evaluation SF eval: stockfish_eval_cp_clipped (±1000 cp)


In [3]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C3:    DATA PIPELINE                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
PIECE_VALUES = {chess.PAWN:1, chess.KNIGHT:3, chess.BISHOP:3, chess.ROOK:5, chess.QUEEN:9}

def board_to_tensor(board):
    t = torch.zeros(12, 8, 8); flip = (board.turn == chess.BLACK)
    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece is None: continue
        row, col = divmod(sq, 8)
        if flip: row = 7 - row
        ch = (0 if piece.color == board.turn else 6) + (piece.piece_type - 1)
        t[ch, row, col] = 1.0
    return t

def apply_move(board, move):
    b = board.copy(); b.push(move); return b

def compute_material_diff_pawns(board):
    w = sum(PIECE_VALUES.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.WHITE)
    b = sum(PIECE_VALUES.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.BLACK)
    return abs(w - b)

def compute_material_imbalance(board):
    w = sum(PIECE_VALUES.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.WHITE)
    b = sum(PIECE_VALUES.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.BLACK)
    diff = (w - b) if board.turn == chess.WHITE else (b - w)
    return max(-1.0, min(1.0, diff / 5.0))

def compute_game_phase(board):
    return len(board.piece_map()) / 32.0

def compute_legal_move_count(board):
    return min(len(list(board.legal_moves)) / 60.0, 1.0)

def compute_king_safety(board):
    ksq = board.king(board.turn)
    if ksq is None: return 0.0
    adj = [sq for sq in chess.SQUARES if chess.square_distance(ksq, sq) <= 1 and sq != ksq]
    if not adj: return 0.0
    return sum(1 for sq in adj if board.is_attacked_by(not board.turn, sq)) / len(adj)

def classify_context(board):
    if board.is_game_over(): return None
    md = compute_material_diff_pawns(board)
    nl = len(list(board.legal_moves))
    np_ = len(board.piece_map())
    if md >= 1.5:              return 'A'
    if nl <= 21 and md < 1.5:  return 'C'
    if md < 0.5 and np_ >= 20: return 'B'
    return None

def extract_triples(pgn_path, max_games, target):
    triples = []; gp = 0; lo, hi = MOVE_RANGE
    with open(pgn_path, 'r', errors='replace') as f:
        while gp < max_games and len(triples) < target:
            game = chess.pgn.read_game(f)
            if game is None: break
            gp += 1
            if gp % 5000 == 0: print(f"  {gp:,} games, {len(triples):,} triples...")
            moves = list(game.mainline_moves())
            if len(moves) < lo + 1: continue
            valid = list(range(lo, min(hi, len(moves))))
            if not valid: continue
            samp = sorted(random.sample(valid, min(POSITIONS_PER_GAME, len(valid))))
            board = game.board()
            for mi, move in enumerate(moves):
                if mi in samp:
                    fb = board.fen(); bc = board.copy(); bc.push(move)
                    triples.append((fb, move.uci(), bc.fen()))
                board.push(move)
                if len(triples) >= target: break
    return triples, gp

print("=" * 60 + "\nDATA PIPELINE\n" + "=" * 60)
assert os.path.exists(PGN_PATH)
triples, n_games = extract_triples(PGN_PATH, MAX_GAMES_TO_PARSE, N_POSITIONS)
print(f"Extracted {len(triples):,} triples from {n_games:,} games")

random.shuffle(triples)
split = int(len(triples) * 0.9)
train_triples = triples[:split]
val_triples   = triples[split:]
train_fens_before = [t[0] for t in train_triples]
print(f"Train: {len(train_triples):,} | Val: {len(val_triples):,}")

print("\nClassifying contexts...")
context_fens = {'A': [], 'B': [], 'C': []}
TARGET_PER_CTX = TRAIN_PER_CONTEXT + EVAL_PER_CONTEXT
all_fens = list(set(train_fens_before)); np.random.shuffle(all_fens)
for fen in all_fens:
    ctx = classify_context(chess.Board(fen))
    if ctx and len(context_fens[ctx]) < TARGET_PER_CTX:
        context_fens[ctx].append(fen)
    if all(len(v) >= TARGET_PER_CTX for v in context_fens.values()): break

context_train = {}; context_eval = {}
for ctx in ['A', 'B', 'C']:
    fens = context_fens[ctx]
    n_tr = min(TRAIN_PER_CONTEXT, len(fens) - EVAL_PER_CONTEXT)
    n_ev = min(EVAL_PER_CONTEXT,  len(fens) - n_tr)
    context_train[ctx] = fens[:n_tr]
    context_eval[ctx]  = fens[n_tr:n_tr + n_ev]
    assert len(set(context_train[ctx]) & set(context_eval[ctx])) == 0
    print(f"  {ctx}: {len(context_train[ctx])} train, {len(context_eval[ctx])} eval")

print("\n✓ C3 complete")

DATA PIPELINE
  5,000 games, 14,959 triples...
  10,000 games, 29,925 triples...
  15,000 games, 44,898 triples...
  20,000 games, 59,851 triples...
  25,000 games, 74,843 triples...
  30,000 games, 89,789 triples...
  35,000 games, 104,747 triples...
  40,000 games, 119,706 triples...
  45,000 games, 134,681 triples...
  50,000 games, 149,635 triples...
Extracted 149,638 triples from 50,000 games
Train: 134,674 | Val: 14,964

Classifying contexts...
  A: 1000 train, 1000 eval
  B: 1000 train, 1000 eval
  C: 781 train, 1000 eval

✓ C3 complete


## Frozen Encoder — The Supplied Substrate

The encoder is a frozen 5-head CNN producing an 8D board representation (win probability, material balance, game phase, mobility, king safety). IBF operates on whatever geometry this encoder produces. The encoder is **not** part of the IBF mechanism.

The paper (§5.3): *"The latent space is 12-dimensional: an 8-dimensional frozen board representation from a 5-head CNN concatenated with a 4-dimensional move-feature vector."*

If no pre-trained checkpoint is available, the encoder is retrained from scratch on Stockfish-labeled positions. The GO/NO-GO gate verifies that the encoder has sufficient geometric spread (PC1 < 50%, d90 ≥ 3) and predictive signal (ρ ≥ 0.25) before proceeding.

In [4]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C4: FROZEN ENCODER                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Encoder architecture and training are UNCHANGED per directive.
# The encoder is a frozen substrate; IBF operates on whatever geometry emerges.
# Weak context separation is the input condition — IBF's agency and crucible
# handle it organically. σ calibration happens in C5.
print("=" * 70 + "\n  C4 — FROZEN ENCODER\n" + "=" * 70)

class RHat_C(nn.Module):
    def __init__(self, z_dim=8, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(12, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU())
        self.fc_z = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, z_dim), nn.BatchNorm1d(z_dim))
        self.head_wp  = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
        self.head_mat = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, 1), nn.Tanh())
        self.head_ph  = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
        self.head_mob = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
        self.head_ks  = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())

    def forward(self, x):
        feat = self.conv(x).flatten(1); z = self.fc_z(feat)
        return (self.head_wp(z).squeeze(-1), self.head_mat(z).squeeze(-1),
                self.head_ph(z).squeeze(-1), self.head_mob(z).squeeze(-1),
                self.head_ks(z).squeeze(-1), z)

    def encode(self, x):
        return self.fc_z(self.conv(x).flatten(1))

    def evaluate(self, x):
        return self.head_wp(self.fc_z(self.conv(x).flatten(1))).squeeze(-1)


model_C = RHat_C(z_dim=ENCODER_Z_DIM, dropout=ENCODER_DROPOUT).to(DEVICE)
encoder_loaded = False

for path in [DEV_ENCODER_PATH,
             '/workspace/ibf_8x8/model_C_v2_multitask_best.pt',
             '/workspace/ibf_8x8/model_C_v3_decorr_final.pt',
             os.path.join(CHECKPOINT_DIR, 'encoder_best.pt')]:
    if os.path.exists(path):
        try:
            model_C.load_state_dict(torch.load(path, weights_only=True, map_location=DEVICE))
            model_C.eval(); encoder_loaded = True
            print(f"  Loaded: {path}"); break
        except Exception as e:
            print(f"  Failed {path}: {e}")

if not encoder_loaded:
    print("  Retraining encoder...")
    sample_fens = random.sample(list(set(train_fens_before)),
                                min(ENCODER_N_POSITIONS, len(set(train_fens_before))))
    labeled_data = []; t0 = time.time()

    # BUG 4 FIX: local engine, closed when done.
    _sf_enc = get_sf_engine(threads=1, hash_mb=128)
    try:
        for i, fen in enumerate(sample_fens):
            board = chess.Board(fen)
            if board.is_game_over(): continue
            try:
                info  = _sf_enc.analyse(board, chess.engine.Limit(depth=ENCODER_SF_DEPTH))
                score = info["score"].relative
                if score.is_mate():
                    wp = 1.0 if score.mate() > 0 else 0.0
                else:
                    cp = max(-1500, min(1500, score.score()))
                    wp = 1.0 / (1.0 + np.exp(-cp / 400.0))
            except:
                wp = 0.5
            labeled_data.append((fen, wp))
            if (i + 1) % 3000 == 0:
                print(f"    {i+1}/{len(sample_fens)} ({(i+1)/(time.time()-t0):.0f}/s)")
    finally:
        _sf_enc.quit()

    print(f"  Labeled {len(labeled_data)} in {time.time()-t0:.0f}s")
    random.shuffle(labeled_data); nv = int(len(labeled_data) * 0.15)

    class _DS(Dataset):
        def __init__(s, d): s.d = d
        def __len__(s): return len(s.d)
        def __getitem__(s, i):
            fen, wp = s.d[i]; b = chess.Board(fen)
            return (board_to_tensor(b),
                    torch.tensor(wp,                          dtype=torch.float32),
                    torch.tensor(compute_material_imbalance(b), dtype=torch.float32),
                    torch.tensor(compute_game_phase(b),         dtype=torch.float32),
                    torch.tensor(compute_legal_move_count(b),   dtype=torch.float32),
                    torch.tensor(compute_king_safety(b),        dtype=torch.float32))

    tld = DataLoader(_DS(labeled_data[nv:]),  batch_size=128, shuffle=True)
    vld = DataLoader(_DS(labeled_data[:nv]),  batch_size=128, shuffle=False)
    _mse = nn.MSELoss(); go = False

    for att in range(5):
        print(f"  Attempt {att+1}/5")
        model_C = RHat_C(z_dim=ENCODER_Z_DIM, dropout=ENCODER_DROPOUT).to(DEVICE)
        opt = optim.Adam(model_C.parameters(), lr=ENCODER_LR, weight_decay=1e-4)
        br, be, pat = -1.0, 0, 0; t0 = time.time()

        for ep in range(ENCODER_EPOCHS):
            model_C.train()
            for X, yw, ym, yp, ymob, yks in tld:
                X, yw, ym, yp, ymob, yks = (X.to(DEVICE), yw.to(DEVICE), ym.to(DEVICE),
                                              yp.to(DEVICE), ymob.to(DEVICE), yks.to(DEVICE))
                opt.zero_grad()
                pw, pm, pp, pmob, pks, z = model_C(X)
                l = (_mse(pw, yw) + _mse(pm, ym) + _mse(pp, yp)
                     + _mse(pmob, ymob) + _mse(pks, yks))
                zc = z - z.mean(0); cov = (zc.T @ zc) / (z.shape[0] - 1)
                std = torch.sqrt(torch.diag(cov) + 1e-6)
                corr = cov / (std.unsqueeze(0) * std.unsqueeze(1))
                mask = 1.0 - torch.eye(z.shape[1], device=z.device)
                l += ENCODER_LAMBDA_DECORR * (corr * mask).pow(2).mean()
                l.backward(); opt.step()

            model_C.eval(); vp, vt = [], []
            with torch.no_grad():
                for X, yw, *_ in vld:
                    vp.extend(model_C(X.to(DEVICE))[0].cpu().numpy())
                    vt.extend(yw.numpy())
            rho, _ = scipy_stats.spearmanr(vp, vt)
            if rho > br:
                br, be, pat = rho, ep + 1, 0
                torch.save(model_C.state_dict(), os.path.join(CHECKPOINT_DIR, 'encoder_best.pt'))
            else:
                pat += 1
            if (ep + 1) % 10 == 0:
                print(f"    Ep {ep+1}: rho={rho:.3f} {'*' if pat==0 else ''} [{time.time()-t0:.0f}s]")
            if pat >= ENCODER_PATIENCE:
                print(f"    Early stop"); break

        model_C.load_state_dict(torch.load(
            os.path.join(CHECKPOINT_DIR, 'encoder_best.pt'),
            weights_only=True, map_location=DEVICE))
        model_C.eval()

        with torch.no_grad():
            az = []
            for X, *_ in vld:
                az.append(model_C(X.to(DEVICE))[-1].cpu().numpy())
            az = np.concatenate(az)

        ev  = np.maximum(np.linalg.eigvalsh(np.cov(az - az.mean(0), rowvar=False))[::-1], 0)
        cv  = np.cumsum(ev) / (np.sum(ev) + 1e-10)
        pc1 = cv[0]; d90 = int(np.searchsorted(cv, 0.90)) + 1
        print(f"    PC1={100*pc1:.1f}%, d90={d90}, rho_wp={br:.3f}")
        # Gate: geometry must be distributed (pc1<0.50, d90>=3) AND
        # encoder must have predictive signal (rho>=0.25).
        # rho<0.25 means the encoder is mostly guessing win-probability;
        # D-signals will lack spatial coherence and force learning from scratch.
        if pc1 < 0.50 and d90 >= 3 and br >= 0.25:
            go = True; print(f"    GO: geometry and predictive thresholds met"); break
        else:
            reasons = []
            if pc1 >= 0.50:  reasons.append(f"PC1={100*pc1:.1f}%>=50%")
            if d90 < 3:      reasons.append(f"d90={d90}<3")
            if br < 0.25:    reasons.append(f"rho={br:.3f}<0.25")
            print(f"    NO-GO: {', '.join(reasons)} — retrying")

    assert go, "Encoder GO/NO-GO failed"

model_C.eval()
for p in model_C.parameters(): p.requires_grad = False


@torch.no_grad()
def RC_encode(board):
    return model_C.encode(board_to_tensor(board).unsqueeze(0).to(DEVICE)).squeeze(0).cpu().numpy()

@torch.no_grad()
def RC_R_field(board):
    return float(model_C.evaluate(board_to_tensor(board).unsqueeze(0).to(DEVICE)).item())

def RC_encode_move(board_before, board_after, move):
    z  = RC_encode(board_after)
    pt = board_before.piece_type_at(move.from_square)
    if pt is None: pt = 1
    mf = np.array([move.from_square / 63.0, move.to_square / 63.0,
                   pt / 6.0, float(board_before.is_capture(move))]) * MOVE_SCALE
    return np.concatenate([z, mf])

@torch.no_grad()
def RC_encode_batch(boards):
    tensors = torch.stack([board_to_tensor(b) for b in boards]).to(DEVICE)
    return model_C.encode(tensors).cpu().numpy()

@torch.no_grad()
def RC_R_field_batch(boards):
    tensors = torch.stack([board_to_tensor(b) for b in boards]).to(DEVICE)
    return model_C.evaluate(tensors).cpu().numpy()

print(f"  z={np.round(RC_encode(chess.Board()), 3)}, R={RC_R_field(chess.Board()):.4f}")
print("✓ C4 complete")

  C4 — FROZEN ENCODER
  Failed /workspace/ibf_8x8/model_C_v2_GO_backup.pt: Error(s) in loading state_dict for RHat_C:
	Missing key(s) in state_dict: "head_mat.0.weight", "head_mat.0.bias", "head_mat.2.weight", "head_mat.2.bias", "head_ph.0.weight", "head_ph.0.bias", "head_ph.2.weight", "head_ph.2.bias", "head_mob.0.weight", "head_mob.0.bias", "head_mob.2.weight", "head_mob.2.bias", "head_ks.0.weight", "head_ks.0.bias", "head_ks.2.weight", "head_ks.2.bias". 
	Unexpected key(s) in state_dict: "head_material.0.weight", "head_material.0.bias", "head_material.2.weight", "head_material.2.bias", "head_phase.0.weight", "head_phase.0.bias", "head_phase.2.weight", "head_phase.2.bias", "head_mobility.0.weight", "head_mobility.0.bias", "head_mobility.2.weight", "head_mobility.2.bias", "head_king_safety.0.weight", "head_king_safety.0.bias", "head_king_safety.2.weight", "head_king_safety.2.bias". 
  Failed /workspace/ibf_8x8/model_C_v2_multitask_best.pt: Error(s) in loading state_dict for RHat_C:
	M

## Geometric Resolution Calibration (§5.1)

All geometry parameters are derived from the encoder's actual latent space — no grid search.

The calibration principle: σ* must be small enough that a correction deposited at one configuration does not destructively activate at a semantically distinct neighbor. Specifically:

- **σ_train = P10/3.0**: At the 10th-percentile nearest-neighbor distance in 12D, kernel overlap ≤ 1%.
- **σ_floor = sib_median/20**: After thermodynamic shrink, boundary crystals compress toward this floor.
- **σ_agency = d8_p10/1.8**: Responsiveness channel operates in the 8D board-only space.
- **κ = σ*/√d_eff**: The empirical scaling ratio (Table in §5.1).

The paper reports: σ* = 1.2693, d_eff = 4.9, κ = 0.5747.

In [5]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C5 — σ CALIBRATION                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ══════════════════════════════════════════════════════════════════════════════
# Derives all geometry parameters from the encoder's actual latent space.
#
# Derivation of P10/3:
#   At distance d from a memory center, kernel overlap = exp(-d²/2σ²).
#   We want ≤1% bleed at the 10th-percentile nearest-neighbor boundary (P10).
#   exp(-(P10)²/2σ²) ≤ 0.01  →  σ ≤ P10/sqrt(2·ln100) ≈ P10/3.03
#   We use P10/3.0 as the generalization anchor (≈1% bleed at P10 boundary).
#
# SIGMA_FLOOR = sib_median/20 (action quarantine):
#   After thermodynamic shrink, boundary crystals compress toward this floor.
#   Produces floor ≈ 0.15, matching dev-run geometry.
#
# SIGMA_AGENCY = d8_p10/1.8:
#   8D agency space scale, produces ≈ 0.78, matching dev-run geometry.
#
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70 + "\n  C5 — σ CALIBRATION\n" + "=" * 70)

n_cal    = min(2000, len(train_fens_before))
cal_fens = random.sample(list(set(train_fens_before)), n_cal)
cal_z_8d = np.array([RC_encode(chess.Board(fen)) for fen in cal_fens])

# 12D move-augmented vectors for SIGMA_TRAIN calibration
cal_z_12d = []
for fen in cal_fens[:500]:
    board = chess.Board(fen); legal = list(board.legal_moves)
    if len(legal) < 2: continue
    m = random.choice(legal)
    cal_z_12d.append(RC_encode_move(board, apply_move(board, m), m))
cal_z_12d = np.array(cal_z_12d)

# P10 of pairwise 12D distances
i1 = np.random.randint(0, len(cal_z_12d), 5000)
i2 = np.random.randint(0, len(cal_z_12d), 5000)
mask = i1 != i2
dists_12d = np.linalg.norm(cal_z_12d[i1[mask]] - cal_z_12d[i2[mask]], axis=1)
p10 = float(np.percentile(dists_12d, 10))

# ── SIGMA_TRAIN from P10/3 ────────────────────────────────────────────────────
SIGMA_TRAIN     = p10 / 3.0
MERGE_THRESHOLD = SIGMA_TRAIN * 1.5

# ── Sibling distances for SIGMA_FLOOR (action quarantine) ─────────────────────
sib_dists = []
for fen in cal_fens[:200]:
    board = chess.Board(fen); legal = list(board.legal_moves)
    if len(legal) < 2: continue
    zs = [RC_encode_move(board, apply_move(board, m), m) for m in legal[:10]]
    for i in range(len(zs)):
        for j in range(i + 1, min(i + 5, len(zs))):
            sib_dists.append(np.linalg.norm(zs[i] - zs[j]))
sib_median = float(np.median(sib_dists))
sib_p25    = float(np.percentile(sib_dists, 25))
SIGMA_FLOOR = sib_median / 20.0

# ── SIGMA_AGENCY from 8D space ─────────────────────────────────────────────────
d8 = np.linalg.norm(
    cal_z_8d[np.random.randint(0, len(cal_z_8d), 3000)] -
    cal_z_8d[np.random.randint(0, len(cal_z_8d), 3000)], axis=1)
SIGMA_AGENCY = float(np.percentile(d8, 10)) / 1.8

# ── Effective dimensionality and κ ────────────────────────────────────────────
eig_12d   = np.sort(np.var(cal_z_12d, axis=0))[::-1]
d_eff_12d = float((np.sum(eig_12d) ** 2) / np.sum(eig_12d ** 2))
kappa_chess = SIGMA_TRAIN / np.sqrt(d_eff_12d)

# ── Bleed analysis ────────────────────────────────────────────────────────────
D_BC = 3.04   # d(B,C) — the tightest cross-context gap (fresh encoder)
bleed_train = np.exp(-D_BC**2 / (2 * SIGMA_TRAIN**2))
bleed_floor = np.exp(-D_BC**2 / (2 * SIGMA_FLOOR**2))

# ── Geometric capacity check ──────────────────────────────────────────────────
centers_est = [cal_z_12d[0]]
for i in range(1, len(cal_z_12d)):
    d = np.array([np.linalg.norm(cal_z_12d[i] - c) for c in centers_est])
    if np.all(d >= MERGE_THRESHOLD): centers_est.append(cal_z_12d[i])
    if len(centers_est) > 5000: break
geo_capacity = len(centers_est)

# ── Report ─────────────────────────────────────────────────────────────────────
print(f"  P10={p10:.4f}")
print(f"  SIGMA_TRAIN   = P10/3.0   = {SIGMA_TRAIN:.4f}  (1% bleed at P10 boundary)")
print(f"  MERGE         = σ×1.5     = {MERGE_THRESHOLD:.4f}")
print(f"  Sibling median={sib_median:.4f}, P25={sib_p25:.4f}")
print(f"  SIGMA_FLOOR   = sib/20.0  = {SIGMA_FLOOR:.4f}  (action quarantine)")
print(f"  SIGMA_AGENCY  = d8p10/1.8 = {SIGMA_AGENCY:.4f}")
print(f"  d_eff={d_eff_12d:.1f}, kappa_chess={kappa_chess:.4f}")
print(f"")
print(f"  ── Bleed analysis ───────────────────────────────────────────────")
print(f"  d(B,C) = {D_BC:.2f} (tightest cross-context gap, fresh encoder)")
print(f"  Bleed at σ_train={SIGMA_TRAIN:.3f}: {100*bleed_train:.3f}%  "
      f"({'fatal for MLP/replay' if bleed_train > 0.04 else 'marginal'})")
print(f"  Bleed at σ_floor={SIGMA_FLOOR:.3f}: {bleed_floor*100:.8f}%  "
      f"(post-Crucible quarantine)")
print(f"  Crucible compression factor: {bleed_train/bleed_floor:.2e}x")
print(f"")
print(f"  Geometric capacity: ~{geo_capacity} non-overlapping kernel volumes")

# Sanity check: warn if σ drifted far from reference value
ref = 1.22
pct = abs(SIGMA_TRAIN - ref) / ref * 100
if pct > 20:
    print(f"  ⚠ σ={SIGMA_TRAIN:.4f} differs from reference 1.22 by {pct:.0f}% "
          f"— geometry has shifted. STOP and report if >25%.")
elif pct > 10:
    print(f"  NOTE: σ={SIGMA_TRAIN:.4f} differs from reference 1.22 by {pct:.1f}%"
          f" — within tolerance, proceeding")
else:
    print(f"  ✓ σ consistent with reference 1.22 (Δ={pct:.1f}%)")

# Reference target check for floor and agency
ref_floor  = 0.1364
ref_agency = 0.7549
pct_floor  = abs(SIGMA_FLOOR - ref_floor) / ref_floor * 100
pct_agency = abs(SIGMA_AGENCY - ref_agency) / ref_agency * 100
print(f"  ✓ σ_floor  vs ref 0.1364: Δ={pct_floor:.1f}%"
      f"{'  ⚠ >25%' if pct_floor > 25 else ''}")
print(f"  ✓ σ_agency vs ref 0.7549: Δ={pct_agency:.1f}%"
      f"{'  ⚠ >25%' if pct_agency > 25 else ''}")

print("\n✓ C5 complete")

  C5 — σ CALIBRATION
  P10=3.8080
  SIGMA_TRAIN   = P10/3.0   = 1.2693  (1% bleed at P10 boundary)
  MERGE         = σ×1.5     = 1.9040
  Sibling median=2.7247, P25=1.5745
  SIGMA_FLOOR   = sib/20.0  = 0.1362  (action quarantine)
  SIGMA_AGENCY  = d8p10/1.8 = 0.7806
  d_eff=4.9, kappa_chess=0.5747

  ── Bleed analysis ───────────────────────────────────────────────
  d(B,C) = 3.04 (tightest cross-context gap, fresh encoder)
  Bleed at σ_train=1.269: 5.682%  (fatal for MLP/replay)
  Bleed at σ_floor=0.136: 0.00000000%  (post-Crucible quarantine)
  Crucible compression factor: 7.51e+106x

  Geometric capacity: ~282 non-overlapping kernel volumes
  ✓ σ consistent with reference 1.22 (Δ=4.0%)
  ✓ σ_floor  vs ref 0.1364: Δ=0.1%
  ✓ σ_agency vs ref 0.7549: Δ=3.4%

✓ C5 complete


In [6]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C6: IBF ENGINE                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ══════════════════════════════════════════════════════════════════════════════
# IBFParams defaults updated to match locked run parameters:
#   reversal_threshold: -0.0085 (was -0.0125)
#   alpha_shrink: 500.0 (was 100.0)
# These defaults are overridden by build_params() in C8 anyway, but
# having correct defaults prevents silent bugs if IBFParams is ever
# instantiated without build_params.
# ══════════════════════════════════════════════════════════════════════════════

from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class IBFParams:
    sigma:float=1.0; merge_threshold:float=1.5; sigma_agency:float=None
    eta:float=0.1; eta_cryst:float=0.005; eta_k:float=0.10
    mu_base:float=0.06; mu_crystallized:float=0.001
    v_max:float=0.50; w_max:float=5.0; k_0:float=5.0; k_min:float=1.0
    crystallization_threshold:int=15; convergence_threshold:float=0.025
    n_cross_min:int=8; reversal_threshold:float=-0.0085        # ← locked param
    w_dvar_threshold:float=0.005; activation_threshold:float=0.01
    creation_threshold:float=0.01; capacity:int=50000
    alpha_shrink:float=500.0; sigma_floor:float=0.15           # ← locked param
    min_samples_shrink:int=50

@dataclass
class MemoryCenter:
    z:np.ndarray; v:float=0.0; w:float=0.0; n_updates:int=0
    D_sum:float=0.0; D_sq_sum:float=0.0; mu_eff:float=0.06
    context_id:int=-1; birth_epoch:int=0
    context_update_counts:Dict[int,int]=field(default_factory=dict)
    sigma:float=1.0; D_history:List[float]=field(default_factory=list)
    D_history_cross:List[float]=field(default_factory=list)
    was_ever_crystallized:bool=False; crucible_verified:bool=False
    dissolution_log:List[Dict]=field(default_factory=list)
    def is_crystallized(self): return self.mu_eff < 0.06-1e-6
    def D_var_rolling(self):
        if len(self.D_history)<25: return 0.0
        rec=self.D_history[20:][-50:]; return float(np.var(rec)) if len(rec)>=5 else 0.0

class IBFEngine:
    def __init__(self, params=None):
        self.p=params or IBFParams(); self.value_centers=[]; self.agency_centers=[]
        self.current_epoch=0; self.current_context_id=-1; self._epoch_D_vals=[]
        self._merge_ratio=self.p.merge_threshold/self.p.sigma
        self._dsf=min(self.p.sigma_floor,self.p.sigma*0.25)
        self._nt=self.p.sigma*0.50
        self._sa=self.p.sigma_agency if self.p.sigma_agency is not None else self.p.sigma
        self._Drs=0.0; self._Drc=0
    def _kb(self, z, centers):
        if not centers: return np.array([])
        za=np.array([c.z for c in centers]); sq=np.sum((za-z[np.newaxis,:])**2,axis=1)
        sig=np.array([c.sigma for c in centers]); return np.exp(-sq/(2.0*sig**2))
    def _rg(self, c):
        if c.context_id==self.current_context_id: return 1.0
        if c.is_crystallized() and c.crucible_verified: return 1.0
        return 0.0
    def delta_R(self, z):
        if not self.value_centers: return 0.0
        K=self._kb(z,self.value_centers); t=0.0
        for i,c in enumerate(self.value_centers):
            g=self._rg(c)
            if g>0 and K[i]>self.p.activation_threshold: t+=g*c.v*K[i]
        return t
    def delta_R_batch(self, Z):
        if not self.value_centers: return np.zeros(len(Z))
        Zc=np.array([c.z for c in self.value_centers]); sig=np.array([c.sigma for c in self.value_centers])
        vs=np.array([c.v for c in self.value_centers]); gate=np.array([self._rg(c) for c in self.value_centers])
        sq=np.sum(Z**2,1,keepdims=True)+np.sum(Zc**2,1,keepdims=True).T-2.0*(Z@Zc.T)
        K=np.exp(-sq/(2.0*sig[np.newaxis,:]**2)); K[K<self.p.activation_threshold]=0.0
        return K@(gate*vs)
    def k_eff(self, z):
        if not self.agency_centers: return self.p.k_0
        K=self._kb(z,self.agency_centers); tw,sK=0.0,0.0
        for i,c in enumerate(self.agency_centers):
            if not c.is_crystallized(): continue
            g=self._rg(c)
            if g>0 and K[i]>self.p.activation_threshold: tw+=g*c.w*K[i]; sK+=g*K[i]
        dk=tw/sK if sK>1e-6 else 0.0
        return max(self.p.k_min, self.p.k_0+dk)
    def select_move(self, board, deterministic=False):
        legal=list(board.legal_moves)
        if not legal: return None, {}
        if len(legal)==1: return legal[0], {'forced':True,'k_pos':0.0,'entropy':0.0}
        zc=RC_encode(board); kp=self.k_eff(zc); Rc=RC_R_field(board)
        afters=[apply_move(board,m) for m in legal]
        rf=RC_R_field_batch(afters)
        za=[RC_encode_move(board,afters[i],legal[i]) for i in range(len(legal))]
        Z=np.array(za); dR=self.delta_R_batch(Z)
        Re=np.clip(1.0-(rf+dR),0.0,1.0); dj=Re-Rc
        logits=kp*dj; logits-=np.max(logits); probs=np.exp(logits)/np.sum(np.exp(logits))
        idx=int(np.argmax(probs)) if deterministic else np.random.choice(len(legal),p=probs)
        return legal[idx], {'k_pos':float(kp),'entropy':float(-np.sum(probs*np.log(probs+1e-10)))}
    def compute_D_and_update(self, bb, bao, bop, move_chosen=None, move_opponent=None, R_imposed_override=None):
        zb=RC_encode(bb)
        zch=RC_encode_move(bb,bao,move_chosen) if move_chosen else RC_encode(bao)
        Rfc=RC_R_field(bao); dRc=self.delta_R(zch); Rec=np.clip(1.0-(Rfc+dRc),0.0,1.0)
        if R_imposed_override is not None: Rei=float(R_imposed_override)
        else:
            zr=RC_encode_move(bao,bop,move_opponent) if move_opponent else RC_encode(bop)
            Rei=np.clip(RC_R_field(bop)+self.delta_R(zr),0.0,1.0)
        D=Rei-Rec; self._Drc+=1; self._Drs+=D; D-=self._Drs/self._Drc
        self._epoch_D_vals.append(D); self._uv(zch,D); self._ua(zb,D)
        return {'D':D,'skipped':False}
    def _sh(self, c):
        if len(c.D_history)>=self.p.min_samples_shrink:
            dv=c.D_var_rolling(); c.sigma=min(c.sigma,max(self._dsf,self.p.sigma/(1.0+self.p.alpha_shrink*dv)))
    def _uv(self, z, D):
        nD=-D
        for c in [c for c in self.value_centers if c.is_crystallized() and c.context_id!=self.current_context_id]:
            K=self._kb(z,[c]); kw=float(K[0]) if len(K)>0 else 0.0
            if kw<self.p.activation_threshold: continue
            jD=nD*kw; c.v=np.clip(c.v+self.p.eta_cryst*jD,-self.p.v_max,self.p.v_max)
            c.n_updates+=1; c.D_sum+=jD; c.D_sq_sum+=jD*jD
            c.context_update_counts[self.current_context_id]=c.context_update_counts.get(self.current_context_id,0)+1
            c.D_history_cross.append(nD)
        same=[c for c in self.value_centers if c.context_id==self.current_context_id]
        Ks=self._kb(z,same) if same else np.array([]); maxK=float(np.max(Ks)) if len(Ks)>0 else 0.0
        if maxK<self.p.creation_threshold:
            if len(self.value_centers)<self.p.capacity:
                jD=nD; nc=MemoryCenter(z=z.copy(),v=np.clip(self.p.eta*jD,-self.p.v_max,self.p.v_max),
                    mu_eff=self.p.mu_base,context_id=self.current_context_id,birth_epoch=self.current_epoch,sigma=self.p.sigma)
                nc.n_updates=1;nc.D_sum=jD;nc.D_sq_sum=jD*jD;nc.D_history.append(jD)
                nc.context_update_counts[self.current_context_id]=1;self.value_centers.append(nc)
            return
        for i,c in enumerate(same):
            kw=float(Ks[i])
            if kw<self.p.activation_threshold: continue
            jD=nD*kw; lr=self.p.eta_cryst if c.is_crystallized() else self.p.eta
            c.v=np.clip(c.v+lr*jD,-self.p.v_max,self.p.v_max)
            c.n_updates+=1;c.D_sum+=jD;c.D_sq_sum+=jD*jD
            c.context_update_counts[self.current_context_id]=c.context_update_counts.get(self.current_context_id,0)+1
            c.D_history.append(jD);self._sh(c)
    def _ua(self, z, D):
        for c in [c for c in self.agency_centers if c.is_crystallized() and c.context_id!=self.current_context_id]:
            K=self._kb(z,[c]); kw=float(K[0]) if len(K)>0 else 0.0
            if kw<self.p.activation_threshold: continue
            c.n_updates+=1;c.context_update_counts[self.current_context_id]=c.context_update_counts.get(self.current_context_id,0)+1
            c.D_history_cross.append(D)
        same=[c for c in self.agency_centers if c.context_id==self.current_context_id]
        Ks=self._kb(z,same) if same else np.array([]); maxK=float(np.max(Ks)) if len(Ks)>0 else 0.0
        if maxK<self.p.creation_threshold:
            if len(self.agency_centers)<self.p.capacity:
                nc=MemoryCenter(z=z.copy(),mu_eff=self.p.mu_base,context_id=self.current_context_id,
                    birth_epoch=self.current_epoch,sigma=self._sa)
                nc.n_updates=1;nc.D_sum=D;nc.D_sq_sum=D*D;nc.D_history.append(D)  # ← BUG 2 FIX: D_history.append present at creation
                nc.context_update_counts[self.current_context_id]=1;self.agency_centers.append(nc)
            return
        for i,c in enumerate(same):
            kw=float(Ks[i])
            if kw<self.p.activation_threshold: continue
            jD=D*kw;c.n_updates+=1;c.D_sum+=jD;c.D_sq_sum+=jD*jD
            c.context_update_counts[self.current_context_id]=c.context_update_counts.get(self.current_context_id,0)+1
            c.D_history.append(jD)  # ← BUG 2 FIX: this line was missing from the transcription
            if c.is_crystallized():
                dv=c.D_var_rolling(); tw=np.clip(self.p.w_max*(1.0-dv/self.p.w_dvar_threshold),-self.p.w_max,self.p.w_max)
                c.w+=self.p.eta_k*kw*(tw-c.w); c.w=np.clip(c.w,-self.p.w_max,self.p.w_max)
            self._sh(c)
    def end_epoch(self):
        for pop in [self.value_centers,self.agency_centers]:
            for c in pop: c.v*=(1.0-c.mu_eff); c.w*=(1.0-c.mu_eff)
        for pop in [self.value_centers,self.agency_centers]:
            for c in pop:
                if c.is_crystallized() or c.n_updates<self.p.crystallization_threshold or len(c.D_history)<5: continue
                if abs(float(np.mean(c.D_history[-50:])))<self.p.convergence_threshold:
                    c.mu_eff=self.p.mu_crystallized; c.was_ever_crystallized=True
        nv,nd=0,0
        for pop,uw in [(self.value_centers,False),(self.agency_centers,True)]:
            for c in pop:
                if not c.is_crystallized(): continue
                ncr=len(c.D_history_cross)
                if ncr<self.p.n_cross_min: continue
                mu=float(np.mean(c.D_history_cross[-min(ncr,50):]))
                if not uw: prod,th=c.v*mu,self.p.reversal_threshold
                else: prod,th=c.w*-abs(mu),self.p.reversal_threshold*(self.p.w_max/self.p.v_max)
                if prod<th:
                    c.dissolution_log.append({'step':self.current_epoch,'v':float(c.v),'w':float(c.w),'mu':float(mu),'prod':float(prod),'ncr':ncr})
                    c.mu_eff=self.p.mu_base;c.crucible_verified=False;nd+=1
                else: c.crucible_verified=True;nv+=1
        for attr in ['value_centers','agency_centers']:
            centers=getattr(self,attr)
            if len(centers)<2: continue
            merged=set();new=[]
            for i in range(len(centers)):
                if i in merged: continue
                best=centers[i]
                for j in range(i+1,len(centers)):
                    if j in merged or centers[i].context_id!=centers[j].context_id: continue
                    d=np.linalg.norm(centers[i].z-centers[j].z)
                    ni,nj=centers[i].sigma<self._nt,centers[j].sigma<self._nt
                    dt=self._merge_ratio*(max(centers[i].sigma,centers[j].sigma)*1.5 if ni and nj else min(centers[i].sigma,centers[j].sigma))
                    if d<dt:
                        o=centers[j]
                        if o.n_updates>best.n_updates: best,o=o,best
                        best.v=np.clip(best.v+o.v,-self.p.v_max,self.p.v_max);best.w=np.clip(best.w+o.w,-self.p.w_max,self.p.w_max)
                        best.n_updates+=o.n_updates;best.D_sum+=o.D_sum;best.D_sq_sum+=o.D_sq_sum
                        for k,cnt in o.context_update_counts.items():best.context_update_counts[k]=best.context_update_counts.get(k,0)+cnt
                        best.D_history.extend(o.D_history);best.D_history_cross.extend(o.D_history_cross)
                        best.sigma=min(best.sigma,o.sigma)
                        if o.was_ever_crystallized:best.was_ever_crystallized=True
                        merged.add(j)
                new.append(best)
            if len(new)>self.p.capacity:
                cr=[c for c in new if c.is_crystallized()];tr=sorted([c for c in new if not c.is_crystallized()],key=lambda c:(abs(c.v)+abs(c.w))*c.n_updates)
                nk=self.p.capacity-len(cr);new=cr+tr[-nk:] if nk>0 else cr[:self.p.capacity]
            setattr(self,attr,new)
        D=self._epoch_D_vals
        diag={'epoch':self.current_epoch,'D_abs_mean':float(np.mean(np.abs(D))) if D else 0.0,
              'n_value_centers':len(self.value_centers),'n_value_crystallized':sum(1 for c in self.value_centers if c.is_crystallized()),
              'n_value_verified':sum(1 for c in self.value_centers if c.is_crystallized() and c.crucible_verified),
              'n_agency_centers':len(self.agency_centers),
              'frac_crystallized':sum(1 for c in self.value_centers+self.agency_centers if c.is_crystallized())/max(1,len(self.value_centers)+len(self.agency_centers)),
              'n_verified':nv,'n_dissolved':nd}
        self._epoch_D_vals=[];self.current_epoch+=1;return diag
    def set_context(self, cid): self.current_context_id=cid
    def reset_verifications(self):
        for c in self.value_centers+self.agency_centers: c.crucible_verified=False; c.D_history_cross=[]
    def get_stats(self):
        nv=len(self.value_centers);ncv=sum(1 for c in self.value_centers if c.is_crystallized())
        nvv=sum(1 for c in self.value_centers if c.is_crystallized() and c.crucible_verified)
        return f"Val:{nv}c({ncv}x,{nvv}v)|Agn:{len(self.agency_centers)}c|Ep:{self.current_epoch}"

print("✓ C6 IBF Engine loaded")

✓ C6 IBF Engine loaded


In [7]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL C7: BASELINES & AGENTS                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Agent definitions ───────────────────────────────────────────────────────
class PassiveAgent:
    def __init__(self, k=5.0): self.k = k
    def select_move(self, board, deterministic=False):
        legal = list(board.legal_moves)
        if not legal: return None, {}
        if len(legal) == 1: return legal[0], {'forced': True}
        afters = [apply_move(board, m) for m in legal]
        rf = RC_R_field_batch(afters)
        logits = self.k * (1.0 - rf)
        logits -= np.max(logits); probs = np.exp(logits) / np.sum(np.exp(logits))
        idx = int(np.argmax(probs)) if deterministic else np.random.choice(len(legal), p=probs)
        return legal[idx], {'entropy': float(-np.sum(probs * np.log(probs + 1e-10)))}

class DeltaROnlyAgent:
    def __init__(self, ibf, k=5.0): self.ibf = ibf; self.k = k
    def set_context(self, cid): self.ibf.set_context(cid)
    def select_move(self, board, deterministic=False):
        legal = list(board.legal_moves)
        if not legal: return None, {}
        if len(legal) == 1: return legal[0], {}
        afters = [apply_move(board, m) for m in legal]
        rf = RC_R_field_batch(afters)
        za = [RC_encode_move(board, afters[i], legal[i]) for i in range(len(legal))]
        Ro = np.clip(np.array(rf) + np.array([self.ibf.delta_R(z) for z in za]), 0.0, 1.0)
        logits = self.k * (1.0 - Ro); logits -= np.max(logits)
        probs = np.exp(logits) / np.sum(np.exp(logits))
        idx = int(np.argmax(probs)) if deterministic else np.random.choice(len(legal), p=probs)
        return legal[idx], {}

class MLPCorrection(nn.Module):
    def __init__(self, z_dim=12, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 1), nn.Tanh())
        self.scale = 0.50
    def forward(self, z): return self.net(z).squeeze(-1) * self.scale

class NeuralBaselineAgent:
    def __init__(self, model, k=5.0): self.model = model; self.k = k
    @torch.no_grad()
    def select_move(self, board, deterministic=False):
        legal = list(board.legal_moves)
        if not legal: return None, {}
        if len(legal) == 1: return legal[0], {}
        afters = [apply_move(board, m) for m in legal]
        rf = RC_R_field_batch(afters)
        za = [RC_encode_move(board, afters[i], legal[i]) for i in range(len(legal))]
        corr = self.model(torch.tensor(np.array(za), dtype=torch.float32)).numpy()
        Ro = np.clip(np.array(rf) + corr, 0.0, 1.0)
        logits = self.k * (1.0 - Ro); logits -= np.max(logits)
        probs = np.exp(logits) / np.sum(np.exp(logits))
        idx = int(np.argmax(probs)) if deterministic else np.random.choice(len(legal), p=probs)
        return legal[idx], {}
    def train_on_data(self, zl, rl, Rl, opt):
        if not zl: return 0.0
        self.model.train()
        z   = torch.tensor(np.array(zl), dtype=torch.float32)
        rb  = torch.tensor(np.array(rl), dtype=torch.float32)
        tgt = torch.tensor(np.array(Rl), dtype=torch.float32)
        loss = nn.MSELoss()(rb + self.model(z), tgt)
        opt.zero_grad(); loss.backward(); opt.step()
        return float(loss.item())

class ReplayMLPAgent(NeuralBaselineAgent):
    def __init__(self, model, k=5.0, buf=10000):
        super().__init__(model, k)
        self.bz, self.br, self.bR = [], [], []; self.bs = buf; self.ns = 0
    def _store(self, z, r, R):
        self.ns += 1
        if len(self.bz) < self.bs:
            self.bz.append(z.copy()); self.br.append(r); self.bR.append(R)
        else:
            i = np.random.randint(0, self.ns)
            if i < self.bs: self.bz[i] = z.copy(); self.br[i] = r; self.bR[i] = R
    def train_on_data(self, zl, rl, Rl, opt):
        if not zl: return 0.0
        for z, r, R in zip(zl, rl, Rl): self._store(z, r, R)
        nr = min(len(zl), len(self.bz))
        ri = np.random.choice(len(self.bz), nr, replace=False) if nr > 0 else []
        za = zl + [self.bz[i] for i in ri]
        ra = rl + [self.br[i] for i in ri]
        Ra = Rl + [self.bR[i] for i in ri]
        self.model.train()
        z   = torch.tensor(np.array(za), dtype=torch.float32)
        rb  = torch.tensor(np.array(ra), dtype=torch.float32)
        tgt = torch.tensor(np.array(Ra), dtype=torch.float32)
        loss = nn.MSELoss()(rb + self.model(z), tgt)
        opt.zero_grad(); loss.backward(); opt.step()
        return float(loss.item())

# ── save_engine / load_engine ──────────────────────────────────────────────────
def save_engine(eng, path):
    state = {
        'value_centers': [{
            'z':                     c.z.copy(),
            'v':                     float(c.v),
            'w':                     float(c.w),
            'sigma':                 float(c.sigma),
            'context_id':            int(c.context_id),
            'mu_eff':                float(c.mu_eff),
            'n_updates':             int(c.n_updates),
            'crucible_verified':     bool(c.crucible_verified),
            'was_ever_crystallized': bool(c.was_ever_crystallized),
            'context_update_counts': dict(c.context_update_counts),
            'D_var_snapshot':        float(c.D_var_rolling()),
        } for c in eng.value_centers],
        'agency_centers': [{
            'z':                     c.z.copy(),
            'w':                     float(c.w),
            'sigma':                 float(c.sigma),
            'context_id':            int(c.context_id),
            'mu_eff':                float(c.mu_eff),
            'n_updates':             int(c.n_updates),
            'crucible_verified':     bool(c.crucible_verified),
            'was_ever_crystallized': bool(c.was_ever_crystallized),
            'context_update_counts': dict(c.context_update_counts),
        } for c in eng.agency_centers],
        'meta': {
            'current_epoch':      eng.current_epoch,
            'current_context_id': eng.current_context_id,
            'sigma_train':        eng.p.sigma,
            'sigma_floor':        eng.p.sigma_floor,
        }
    }
    with open(path, 'wb') as f: pickle.dump(state, f)

def load_engine(path, params):
    eng = IBFEngine(params=params)
    with open(path, 'rb') as f:
        state = pickle.load(f)
    for d in state['value_centers']:
        c = MemoryCenter(
            z=d['z'].copy(), v=d['v'], w=d['w'], sigma=d['sigma'],
            context_id=d['context_id'], mu_eff=d['mu_eff'],
            n_updates=d['n_updates'],
            crucible_verified=d['crucible_verified'],
            was_ever_crystallized=d['was_ever_crystallized'])
        c.context_update_counts = dict(d.get('context_update_counts', {}))
        eng.value_centers.append(c)
    for d in state['agency_centers']:
        c = MemoryCenter(
            z=d['z'].copy(), w=d['w'], sigma=d['sigma'],
            context_id=d['context_id'], mu_eff=d['mu_eff'],
            n_updates=d['n_updates'],
            crucible_verified=d['crucible_verified'],
            was_ever_crystallized=d['was_ever_crystallized'])
        c.context_update_counts = dict(d.get('context_update_counts', {}))
        eng.agency_centers.append(c)
    meta = state.get('meta', {})
    if 'current_epoch' in meta: eng.current_epoch = meta['current_epoch']
    print(f"  Loaded {path}: {eng.get_stats()}")
    return eng

# ── Passive baseline ───────────────────────────────────────────────────────────
# Uses stockfish_eval_cp_clipped for evaluation metrics.
# Stores per-position score arrays for downstream persistence.

passive_agent = PassiveAgent(k=1.0)
print("Computing passive baseline...")
passive_cache = {}

_sf_passive = get_sf_engine(threads=1, hash_mb=128)
try:
    for ctx in ['A', 'B', 'C']:
        sc = []
        for fen in context_eval[ctx][:N_EVAL_POSITIONS]:
            board = chess.Board(fen)
            if board.is_game_over(): continue
            m, _ = passive_agent.select_move(board, deterministic=True)
            if m:
                sc.append(-stockfish_eval_cp_clipped(apply_move(board, m), _sf_passive))
        n_clipped = sum(1 for s in sc if abs(s) >= CP_CLIP_EVAL)
        passive_cache[ctx] = {
            'mean_cp':   float(np.mean(sc)),
            'std_cp':    float(np.std(sc)),
            'cp_scores': sc,
        }
        print(f"  Passive {ctx}: {passive_cache[ctx]['mean_cp']:+.1f} cp  "
              f"(std={passive_cache[ctx]['std_cp']:.1f}, clipped={n_clipped})")
finally:
    _sf_passive.quit()

# Gate: clipped mean must be in expected range
expected = {'A': (-400, -250), 'B': (-420, -280), 'C': (-420, -240)}
all_ok = True
for ctx in ['A', 'B', 'C']:
    mean = passive_cache[ctx]['mean_cp']
    lo, hi = expected[ctx]
    if not (lo <= mean <= hi):
        print(f"  ✗ Ctx {ctx}: mean={mean:+.1f} outside [{lo}, {hi}] — resample")
        all_ok = False
if not all_ok:
    raise AssertionError("Passive baseline out of range. Rerun C1-C7.")

# Mate-score contamination check
for ctx in ['A', 'B', 'C']:
    sc = np.array(passive_cache[ctx]['cp_scores'])
    n_outliers = np.sum(np.abs(sc) >= CP_CLIP_EVAL)
    print(f"  Ctx {ctx}: mean={np.mean(sc):+.1f}, "
          f"median={np.median(sc):+.1f}, clipped={n_outliers}")
    if n_outliers > 50:
        print(f"  ⚠ Ctx {ctx} has {n_outliers} clipped scores — eval pool may "
              f"contain too many decisive positions")

print("✓ C7 ready — passive baseline clean, proceed to C8")

Computing passive baseline...
  Passive A: -347.7 cp  (std=274.8, clipped=10)
  Passive B: -360.7 cp  (std=228.6, clipped=4)
  Passive C: -301.8 cp  (std=261.8, clipped=22)
  Ctx A: mean=-347.7, median=-435.0, clipped=10
  Ctx B: mean=-360.7, median=-426.0, clipped=4
  Ctx C: mean=-301.8, median=-302.0, clipped=22
✓ C7 ready — passive baseline clean, proceed to C8


## Pre-Training Diagnostic Gate

Before committing to the full 360-epoch training run, this cell runs a lightweight 10-epoch diagnostic on a throwaway engine. Five binary checks must ALL pass:

1. **Modification dynamics are live**: δR varies across positions
2. **Agency D_history is populated**: BUG 2 fix is working
3. **Agency D_var is nonzero**: Variance-driven responsiveness can crystallize
4. **Cross-context flow exists**: Phase A crystals receive Phase B updates
5. **D_history grows across epochs**: Centers are accumulating interaction history

The critical diagnostic is the **R_opp range check**: if R_opp max ≈ 0.924 instead of ≈ 1.0, the training-path clipping bug has leaked back in.

In [8]:
# C_verify: PRE-TRAINING DIAGNOSTIC GATE
# ══════════════════════════════════════════════════════════════════════════════
# Runs Phase A for 5 epochs + Phase B for 5 epochs on a throwaway engine.
# Five binary checks must ALL pass before C8 is authorized to run.
# If any check fails, the cell raises AssertionError and prints the diagnosis.
#
# Checks:
#   1. delta_R varies across positions (modification dynamics are live)
#   2. At least one agency center has non-empty D_history after Phase A ep 5
#   3. At least one agency center with n_updates > 25 has D_var_rolling() > 0
#   4. After Phase B epoch 1, at least one agency center has
#      cross-context update count > 0 (proves cross-context flow)
#   5. At least one value center has D_history growing across epochs
#
# CRITICAL: Training loop uses stockfish_eval_cp_raw (unclipped).
# The R_opp range diagnostic at the end confirms this — if R_opp_max ≈ 0.924
# instead of ≈ 1.0, the clip leaked into the training path.
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("  C_verify — PRE-TRAINING DIAGNOSTIC GATE")
print("=" * 70)

VERIFY_EPOCHS_A = 5
VERIFY_EPOCHS_B = 5
VERIFY_GAMES    = 20   # lightweight — just enough to populate centers

def build_params_verify():
    p = IBFParams()
    p.sigma              = SIGMA_TRAIN
    p.merge_threshold    = MERGE_THRESHOLD
    p.sigma_agency       = SIGMA_AGENCY
    p.sigma_floor        = SIGMA_FLOOR
    p.v_max              = V_MAX;    p.w_max   = W_MAX
    p.eta                = ETA;      p.eta_cryst = ETA_CRYST; p.eta_k = ETA_K
    p.mu_base            = MU_BASE;  p.mu_crystallized = MU_CRYSTALLIZED
    p.convergence_threshold    = CONVERGENCE_THRESHOLD
    p.crystallization_threshold = CRYSTALLIZATION_THRESHOLD
    p.n_cross_min        = N_CROSS_MIN
    p.reversal_threshold = REVERSAL_THRESHOLD
    p.w_dvar_threshold   = W_DVAR_THRESHOLD
    p.alpha_shrink       = ALPHA_SHRINK
    p.capacity           = CAPACITY
    p.k_0                = K_0
    return p

eng_v  = IBFEngine(params=build_params_verify())
opp_v  = PassiveAgent(k=K_0)

# ── Guard: σ must be set before first game ─────────────────────────────────────
assert SIGMA_TRAIN  > 0,  "SIGMA_TRAIN not set — run C5 first"
assert SIGMA_FLOOR  > 0,  "SIGMA_FLOOR not set — run C5 first"
assert SIGMA_AGENCY > 0,  "SIGMA_AGENCY not set — run C5 first"

_sf_v = get_sf_engine(threads=1, hash_mb=128)

d_history_lengths_by_epoch = {}  # for check 5
all_R_opp_vals = []              # D-signal range diagnostic
all_raw_cp_vals = []             # raw SF score diagnostic

try:
    # ── Phase A: 5 epochs ─────────────────────────────────────────────────────
    eng_v.set_context(0)
    assert eng_v.current_context_id != -1, "set_context failed"
    eng_v.reset_verifications()
    print(f"  Phase A ({VERIFY_EPOCHS_A} epochs)...")

    for ep in range(VERIFY_EPOCHS_A):
        for _ in range(VERIFY_GAMES):
            board = chess.Board(random.choice(context_train['A']))
            for _ in range(MAX_MOVES_PER_GAME):
                if board.is_game_over(): break
                bb = board.copy()
                am, _ = eng_v.select_move(board)
                if am is None: break
                ba = apply_move(board, am); board.push(am)
                if board.is_game_over(): break
                om, _ = opp_v.select_move(board)
                if om is None: break
                bo = apply_move(board, om); board.push(om)
                # CRITICAL: raw, unclipped SF score for training D-signal
                sf_cp  = stockfish_eval_cp_raw(ba, _sf_v, depth=SF_DEPTH_TRAIN)
                R_opp  = 1.0 / (1.0 + np.exp(-sf_cp / 400.0))
                all_raw_cp_vals.append(sf_cp)
                all_R_opp_vals.append(R_opp)
                eng_v.compute_D_and_update(bb, ba, bo, move_chosen=am,
                                           R_imposed_override=1.0 - R_opp)
        diag = eng_v.end_epoch()
        total_dh = sum(len(c.D_history) for c in eng_v.agency_centers)
        d_history_lengths_by_epoch[f'A_{ep+1}'] = total_dh
        if ep == VERIFY_EPOCHS_A - 1:
            print(f"    Ep A-{ep+1}: D={diag['D_abs_mean']:.4f}, "
                  f"val={diag['n_value_centers']}c/{diag['n_value_crystallized']}x, "
                  f"agn={diag['n_agency_centers']}c, "
                  f"agn_D_history_total={total_dh}")

    # ── Phase B: 5 epochs ─────────────────────────────────────────────────────
    eng_v.set_context(1)
    assert eng_v.current_context_id != -1, "set_context failed for Phase B"
    eng_v.reset_verifications()
    print(f"  Phase B ({VERIFY_EPOCHS_B} epochs)...")

    cross_ctx_counts_after_b1 = 0

    for ep in range(VERIFY_EPOCHS_B):
        for _ in range(VERIFY_GAMES):
            board = chess.Board(random.choice(context_train['B']))
            for _ in range(MAX_MOVES_PER_GAME):
                if board.is_game_over(): break
                bb = board.copy()
                am, _ = eng_v.select_move(board)
                if am is None: break
                ba = apply_move(board, am); board.push(am)
                if board.is_game_over(): break
                om, _ = opp_v.select_move(board)
                if om is None: break
                bo = apply_move(board, om); board.push(om)
                # CRITICAL: raw, unclipped SF score for training D-signal
                sf_cp  = stockfish_eval_cp_raw(ba, _sf_v, depth=SF_DEPTH_TRAIN)
                R_opp  = 1.0 / (1.0 + np.exp(-sf_cp / 400.0))
                all_raw_cp_vals.append(sf_cp)
                all_R_opp_vals.append(R_opp)
                eng_v.compute_D_and_update(bb, ba, bo, move_chosen=am,
                                           R_imposed_override=1.0 - R_opp)
        diag = eng_v.end_epoch()
        total_dh = sum(len(c.D_history) for c in eng_v.agency_centers)
        d_history_lengths_by_epoch[f'B_{ep+1}'] = total_dh

        # Check 4: after first B epoch, count cross-context agency updates
        if ep == 0:
            cross_ctx_counts_after_b1 = sum(
                sum(v for k, v in c.context_update_counts.items() if k != 0)
                for c in eng_v.agency_centers)
            print(f"    Ep B-1: D={diag['D_abs_mean']:.4f}, "
                  f"agn cross-ctx updates={cross_ctx_counts_after_b1}, "
                  f"agn_D_history_total={total_dh}")

        if ep == VERIFY_EPOCHS_B - 1:
            print(f"    Ep B-{ep+1}: D={diag['D_abs_mean']:.4f}, "
                  f"val={diag['n_value_centers']}c/{diag['n_value_crystallized']}x, "
                  f"agn={diag['n_agency_centers']}c")

finally:
    _sf_v.quit()

# ── D-signal range diagnostic ──────────────────────────────────────────────────
# This is the single most important diagnostic. It catches the exact bug
# that killed the last run.
print(f"\n  ── D-signal range diagnostic ────────────────────────────────────")
raw_cp = np.array(all_raw_cp_vals)
r_opp  = np.array(all_R_opp_vals)
print(f"  Raw SF cp: range=[{raw_cp.min():.0f}, {raw_cp.max():.0f}], "
      f"mean={raw_cp.mean():.1f}, std={raw_cp.std():.1f}")
print(f"  R_opp:     range=[{r_opp.min():.4f}, {r_opp.max():.4f}], "
      f"mean={r_opp.mean():.4f}, std={r_opp.std():.4f}")
n_mate = int(np.sum(np.abs(raw_cp) >= 10000))
n_over_1000 = int(np.sum(np.abs(raw_cp) > 1000))
print(f"  Positions with |cp| > 1000: {n_over_1000} / {len(raw_cp)}")
print(f"  Positions with mate score (±10000): {n_mate} / {len(raw_cp)}")

# THE CRITICAL CHECK: if R_opp_max ≈ 0.924, clipping leaked into training
if r_opp.max() < 0.95:
    print(f"  ✗✗✗ CRITICAL: R_opp max = {r_opp.max():.4f} < 0.95")
    print(f"      This means CP_CLIP is still in the training path!")
    print(f"      stockfish_eval_cp_raw is NOT being called, or it clips.")
    raise AssertionError("R_opp max < 0.95 — clipping is in the training path. "
                         "Fix stockfish_eval_cp_raw in C2.")
else:
    print(f"  ✓ R_opp max = {r_opp.max():.4f} — training path is unclipped")

# ── Five structural checks ────────────────────────────────────────────────────
print(f"\n  ── Diagnostic checks ────────────────────────────────────────────────")
failures = []

# Check 1: delta_R varies across positions
sample_boards = [chess.Board(f) for f in random.sample(context_train['A'], 10)]
dr_vals = [eng_v.delta_R(RC_encode_move(b,
               apply_move(b, list(b.legal_moves)[0]),
               list(b.legal_moves)[0]))
           for b in sample_boards if list(b.legal_moves)]
dr_std = float(np.std(dr_vals)) if dr_vals else 0.0
check1 = dr_std > 1e-6
print(f"  [{'✓' if check1 else '✗'}] 1. delta_R varies across positions "
      f"(std={dr_std:.6f}, need >1e-6)")
if not check1:
    failures.append("Check 1 FAILED: delta_R is constant — modification dynamics not live. "
                    "Verify _uv D_history append and that value centers are being created.")

# Check 2: at least one agency center has non-empty D_history after Phase A ep 5
agn_with_dh = sum(1 for c in eng_v.agency_centers if len(c.D_history) > 0)
check2 = agn_with_dh > 0
print(f"  [{'✓' if check2 else '✗'}] 2. Agency centers with non-empty D_history: "
      f"{agn_with_dh} / {len(eng_v.agency_centers)}")
if not check2:
    failures.append("Check 2 FAILED: No agency center has D_history populated. "
                    "BUG 2 (_ua D_history append) is still present. Check C6.")

# Check 3: at least one agency center with n_updates>25 has D_var_rolling()>0
heavy_agn = [c for c in eng_v.agency_centers if c.n_updates > 25]
dvar_nonzero = sum(1 for c in heavy_agn if c.D_var_rolling() > 0)
check3 = dvar_nonzero > 0
print(f"  [{'✓' if check3 else '✗'}] 3. Agency centers (n>25) with D_var>0: "
      f"{dvar_nonzero} / {len(heavy_agn)}")
if not check3 and len(heavy_agn) > 0:
    failures.append("Check 3 FAILED: D_var_rolling() is zero for all heavy agency centers. "
                    "D_history is being appended but variance is zero — check D signal magnitude.")

# Check 4: cross-context agency updates > 0 after Phase B epoch 1
check4 = cross_ctx_counts_after_b1 > 0
print(f"  [{'✓' if check4 else '✗'}] 4. Cross-context agency updates after B-ep1: "
      f"{cross_ctx_counts_after_b1} (need >0)")
if not check4:
    failures.append("Check 4 FAILED: No cross-context agency updates in Phase B epoch 1. "
                    "Either no Phase A centers crystallized, or set_context ordering is wrong.")

# Check 5: value center D_history growing across epochs
ep_a1 = d_history_lengths_by_epoch.get('A_1', 0)
ep_a5 = d_history_lengths_by_epoch.get('A_5', 0)
check5 = ep_a5 > ep_a1
print(f"  [{'✓' if check5 else '✗'}] 5. Value D_history grew across epochs "
      f"(A-ep1={ep_a1}, A-ep5={ep_a5})")
if not check5:
    failures.append("Check 5 FAILED: Agency D_history total not growing — "
                    "centers may not be receiving updates.")

# ── Verdict ───────────────────────────────────────────────────────────────────
print()
if not failures:
    print("  ✓ ALL 5 CHECKS PASSED — proceed to C8")
    print(f"    σ_train={SIGMA_TRAIN:.4f}, σ_floor={SIGMA_FLOOR:.4f}, "
          f"σ_agency={SIGMA_AGENCY:.4f}, κ={kappa_chess:.4f}")
    print(f"    Training path confirmed unclipped (R_opp max={r_opp.max():.4f})")
else:
    for msg in failures:
        print(f"  ✗ {msg}")
    raise AssertionError(
        f"{len(failures)}/5 diagnostic checks failed. "
        "Fix the reported issues before running C8.")

del eng_v, opp_v  # discard throwaway engine

  C_verify — PRE-TRAINING DIAGNOSTIC GATE
  Phase A (5 epochs)...
    Ep A-5: D=0.1823, val=237c/117x, agn=57c, agn_D_history_total=7715
  Phase B (5 epochs)...
    Ep B-1: D=0.1482, agn cross-ctx updates=2555, agn_D_history_total=8819
    Ep B-5: D=0.1758, val=465c/245x, agn=106c

  ── D-signal range diagnostic ────────────────────────────────────
  Raw SF cp: range=[-10000, 10000], mean=439.2, std=1460.4
  R_opp:     range=[0.0000, 1.0000], mean=0.6348, std=0.2116
  Positions with |cp| > 1000: 150 / 5982
  Positions with mate score (±10000): 127 / 5982
  ✓ R_opp max = 1.0000 — training path is unclipped

  ── Diagnostic checks ────────────────────────────────────────────────
  [✓] 1. delta_R varies across positions (std=0.003689, need >1e-6)
  [✓] 2. Agency centers with non-empty D_history: 106 / 106
  [✓] 3. Agency centers (n>25) with D_var>0: 94 / 96
  [✓] 4. Cross-context agency updates after B-ep1: 2555 (need >0)
  [✓] 5. Value D_history grew across epochs (A-ep1=1193, A-ep5=7715

## Stage II Training: Full IBF + Baselines

Three-phase sequential training (A → B → C, 120 epochs each) for the full IBF system plus MLP and Replay baselines. All methods receive the same frozen representation, D-signal, and sequential training trajectory (§5.2).

**Critical separation (§6.3):**
- Training D-signal: `stockfish_eval_cp_raw` (unclipped, full ±10000 range)
- Evaluation metrics: `stockfish_eval_cp_clipped` (±1000 cp, symmetric)

The previous development run used a single clipped function for both paths, compressing R_imposed and degrading all downstream IBF mechanisms. This separation is the single most important engineering decision in this notebook.

**Run modes:** `smoke` (10×50, Phase A only), `validate` (30×50, pre-flight checks), `full` (120×100, paper run).

In [11]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Cell C8: TRAINING Full IBF + MLP + Replay baselines                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ══════════════════════════════════════════════════════════════════════════════
# CRITICAL SEPARATION:
#   Training D-signal:  stockfish_eval_cp_raw     (unclipped, full ±10000 range)
#   Evaluation metrics: stockfish_eval_cp_clipped (±1000 cp, symmetric)
#
# SMOKE RUN: set RUN_MODE = 'smoke' for 10-epoch diagnostic before committing.
# VALIDATE:  set RUN_MODE = 'validate' for 30-epoch sanity check.
# FULL:      set RUN_MODE = 'full' for the paper run (120 × 100).
#
# Ablations are in C8b. Full performance eval runs at the end of this cell.
# ══════════════════════════════════════════════════════════════════════════════

RUN_MODE = 'full'   # ← 'smoke' (10×50), 'validate' (30×50), or 'full' (120×100)

if RUN_MODE == 'smoke':
    EPOCHS_RUN = 10;  GAMES_RUN = 50;   EVAL_N = 200
    print("  MODE: SMOKE RUN (10 × 50)")
elif RUN_MODE == 'validate':
    EPOCHS_RUN = 30;  GAMES_RUN = 50;   EVAL_N = 200
    print("  MODE: VALIDATION (30 × 50)")
elif RUN_MODE == 'full':
    EPOCHS_RUN = EPOCHS_PER_PHASE; GAMES_RUN = GAMES_PER_EPOCH; EVAL_N = N_EVAL_TRAINING
    print("  MODE: FULL RUN (120 × 100)")
else:
    raise ValueError(f"RUN_MODE must be 'smoke', 'validate', or 'full', got '{RUN_MODE}'")

# ── Eval helper — clipped, for evaluation ONLY ────────────────────────────────
def eval_agent_on_fens(agent, fens, sf_eng, depth=SF_DEPTH_EVAL):
    """Evaluate agent move quality via Stockfish. Returns clipped scores.
    ONLY for evaluation metrics. NEVER for training D-signals."""
    sc = []
    for fen in fens:
        board = chess.Board(fen)
        if board.is_game_over(): continue
        ctx = classify_context(board)
        cid = {'A': 0, 'B': 1, 'C': 2}.get(ctx, 0) if ctx else 0
        if hasattr(agent, 'set_context'): agent.set_context(cid)
        m, _ = agent.select_move(board, deterministic=True)
        if m:
            sc.append(-stockfish_eval_cp_clipped(apply_move(board, m), sf_eng, depth))
    return np.array(sc)

def eval_agent_on_context(agent, ctx, sf_eng, n=None, depth=SF_DEPTH_EVAL):
    """Evaluate on a single context pool. Returns clipped scores."""
    if n is None: n = EVAL_N
    fens = context_eval[ctx][:n]
    sc = []
    for fen in fens:
        board = chess.Board(fen)
        if board.is_game_over(): continue
        m, _ = agent.select_move(board, deterministic=True)
        if m:
            sc.append(-stockfish_eval_cp_clipped(apply_move(board, m), sf_eng, depth))
    n_clipped = sum(1 for s in sc if abs(s) >= CP_CLIP_EVAL)
    return {'mean_cp': float(np.mean(sc)) if sc else 0.0,
            'cp_scores': sc,
            'n_clipped': n_clipped}

# ── JSON serializer ───────────────────────────────────────────────────────────
def _js(o):
    if isinstance(o, dict):          return {k: _js(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_js(v) for v in o]
    if isinstance(o, np.floating):   return float(o)
    if isinstance(o, np.integer):    return int(o)
    if isinstance(o, np.ndarray):    return o.tolist()
    return o

# ── build_params ──────────────────────────────────────────────────────────────
def build_params(**ov):
    p = IBFParams()
    p.sigma               = SIGMA_TRAIN
    p.merge_threshold     = MERGE_THRESHOLD
    p.sigma_agency        = SIGMA_AGENCY
    p.sigma_floor         = SIGMA_FLOOR
    p.v_max               = V_MAX;   p.w_max     = W_MAX
    p.eta                 = ETA;     p.eta_cryst = ETA_CRYST; p.eta_k = ETA_K
    p.mu_base             = MU_BASE; p.mu_crystallized = MU_CRYSTALLIZED
    p.convergence_threshold     = CONVERGENCE_THRESHOLD
    p.crystallization_threshold = CRYSTALLIZATION_THRESHOLD
    p.n_cross_min         = N_CROSS_MIN
    p.reversal_threshold  = REVERSAL_THRESHOLD
    p.w_dvar_threshold    = W_DVAR_THRESHOLD
    p.alpha_shrink        = ALPHA_SHRINK
    p.capacity            = CAPACITY
    p.k_0                 = K_0
    for k, v in ov.items(): setattr(p, k, v)
    return p

PHASES = [
    {'n': 'A', 'l': 'Tactical/Imbalanced', 'c': 0},
    {'n': 'B', 'l': 'Positional/Quiet',    'c': 1},
    {'n': 'C', 'l': 'Restricted',          'c': 2},
]

# ══════════════════════════════════════════════════════════════════════════════
#  run_training — core training loop (used by C8 and C8b)
# ══════════════════════════════════════════════════════════════════════════════
def run_training(label, ov=None, train_bl=False):
    p   = build_params(**(ov or {}))
    eng = IBFEngine(params=p)
    opp = PassiveAgent(k=K_0)
    ma, ra, mo, ro = None, None, None, None
    if train_bl:
        mm = MLPCorrection(); mo = optim.Adam(mm.parameters(), lr=0.001)
        ma = NeuralBaselineAgent(mm, k=K_0)
        rm = MLPCorrection(); ro = optim.Adam(rm.parameters(), lr=0.001)
        ra = ReplayMLPAgent(rm, k=K_0)

    pea, logs = [], []
    ge = 0
    t0 = time.time()

    # D-signal diagnostics (collected across entire run for smoke/validate)
    all_raw_cp_train = []
    all_R_opp_train  = []

    print(f"\n{'='*70}")
    if p.sigma_agency:
        print(f"  {label} | σ={p.sigma:.4f}, merge={p.merge_threshold:.4f}, σ_agn={p.sigma_agency:.4f}")
    else:
        print(f"  {label} | σ={p.sigma:.4f}")
    if ov: print(f"  Overrides: {ov}")
    print(f"  epochs={EPOCHS_RUN}, games={GAMES_RUN}, eval_n={EVAL_N}")
    print(f"{'='*70}")

    sf_train    = get_sf_engine(threads=1, hash_mb=128)
    sf_eval_loc = get_sf_engine(threads=1, hash_mb=128)

    try:
        for ph in PHASES:
            pn, cid = ph['n'], ph['c']
            pool = context_train[pn]
            print(f"\n{'─'*70}")
            print(f"  PHASE {pn} — {ph['l']} (epochs {ge+1}-{ge+EPOCHS_RUN})")
            print(f"{'─'*70}")

            eng.set_context(cid)
            assert eng.current_context_id != -1, f"set_context failed for phase {pn}"
            eng.reset_verifications()
            print(f"  reset_verifications() called — all crystals must re-earn broadcast")

            for ep in range(EPOCHS_RUN):
                ep_t0 = time.time()
                ez, erb, eR = [], [], []

                for _ in range(GAMES_RUN):
                    board = chess.Board(random.choice(pool))
                    for _ in range(MAX_MOVES_PER_GAME):
                        if board.is_game_over(): break
                        bb = board.copy()
                        am, _ = eng.select_move(board)
                        if am is None: break
                        ba = apply_move(board, am); board.push(am)
                        if board.is_game_over(): break
                        om, _ = opp.select_move(board)
                        if om is None: break
                        bo = apply_move(board, om); board.push(om)

                        # ── TRAINING D-SIGNAL: raw, unclipped ─────────────
                        sf_cp = stockfish_eval_cp_raw(ba, sf_train, SF_DEPTH_TRAIN)
                        R_opp = 1.0 / (1.0 + np.exp(-sf_cp / 400.0))

                        all_raw_cp_train.append(sf_cp)
                        all_R_opp_train.append(R_opp)

                        eng.compute_D_and_update(bb, ba, bo, move_chosen=am,
                                                 R_imposed_override=1.0 - R_opp)
                        ez.append(RC_encode_move(bb, ba, am))
                        erb.append(RC_R_field(ba))
                        eR.append(R_opp)

                if train_bl and ez:
                    ma.train_on_data(ez, erb, eR, mo)
                    ra.train_on_data(ez, erb, eR, ro)

                diag = eng.end_epoch()
                logs.append({'ge': ge, 'ph': pn, **diag})
                ep_elapsed = time.time() - ep_t0

                log_this = (RUN_MODE in ('smoke', 'validate')) or (ep + 1) % 10 == 0 or ep == 0
                if log_this:
                    nv       = diag['n_value_centers']
                    ncv      = diag['n_value_crystallized']
                    nvv      = diag.get('n_value_verified', 0)
                    na       = diag['n_agency_centers']
                    na_cryst = sum(1 for c in eng.agency_centers if c.is_crystallized())
                    nve      = diag.get('n_verified', 0)
                    ndi      = diag.get('n_dissolved', 0)
                    sat      = sum(1 for c in eng.value_centers
                                   if abs(c.v) > 0.95 * p.v_max) / max(1, nv)
                    sigs     = [c.sigma for c in eng.value_centers]
                    sm       = np.mean(sigs) if sigs else p.sigma
                    smin     = np.min(sigs)  if sigs else p.sigma
                    ndl      = sum(1 for s in sigs if s < p.sigma * 0.5)
                    print(f"  Ep {ge+1:3d} ({pn}-{ep+1:3d}): D={diag['D_abs_mean']:.4f}, "
                          f"val={nv}c/{ncv}x/{nvv}v, agn={na}c/{na_cryst}x, "
                          f"sat={100*sat:.0f}%, {ep_elapsed:.1f}s")
                    if nve + ndi > 0:
                        print(f"    Crucible: {nve} verified, {ndi} dissolved")
                    print(f"    σ: mean={sm:.3f}, min={smin:.3f}, needles={ndl}")

                ge += 1

            phase_elapsed = time.time() - t0
            print(f"\n  Phase {pn} complete in {phase_elapsed:.0f}s")
            print(f"  {eng.get_stats()}")

            # ── SMOKE RUN: D-signal diagnostic after Phase A ──────────────
            if RUN_MODE == 'smoke' and pn == 'A':
                rc = np.array(all_raw_cp_train)
                ro_arr = np.array(all_R_opp_train)
                print(f"\n  ── D-signal distribution (Phase A) ─────────────────────")
                print(f"  Raw SF cp: range=[{rc.min():.0f}, {rc.max():.0f}], "
                      f"mean={rc.mean():.1f}, std={rc.std():.1f}")
                print(f"  R_opp:     range=[{ro_arr.min():.4f}, {ro_arr.max():.4f}], "
                      f"mean={ro_arr.mean():.4f}, std={ro_arr.std():.4f}")
                n_over = int(np.sum(np.abs(rc) > 1000))
                n_mate = int(np.sum(np.abs(rc) >= 10000))
                print(f"  |cp| > 1000: {n_over}/{len(rc)}  "
                      f"mate scores: {n_mate}/{len(rc)}")
                if ro_arr.max() < 0.95:
                    print(f"  ✗✗✗ CRITICAL: R_opp max={ro_arr.max():.4f} — clip leaked!")
                    raise AssertionError("Clipping in training path detected in smoke run")
                else:
                    print(f"  ✓ Training path confirmed unclipped")

            # ── Phase-end evaluation ──────────────────────────────────────
            print(f"  Evaluating (n={EVAL_N})...")

            pe = {'phase': pn}
            aev = [('ibf', eng)]
            if train_bl: aev += [('mlp', ma), ('replay', ra)]

            for an, ag in aev:
                pe[an] = {}
                for ctx in ['A', 'B', 'C']:
                    if an == 'ibf':
                        eng.set_context({'A': 0, 'B': 1, 'C': 2}[ctx])
                    pe[an][ctx] = eval_agent_on_context(ag, ctx, sf_eval_loc, n=EVAL_N)

            pe['passive'] = passive_cache
            pea.append(pe)

            print(f"\n  Performance Matrix (After Phase {pn}):")
            print(f"  {'':>12}  {'Ctx A':>10}  {'Ctx B':>10}  {'Ctx C':>10}")
            for an in [k for k, _ in aev] + ['passive']:
                r = [f"{pe[an][c]['mean_cp']:+.1f}" for c in ['A', 'B', 'C']]
                print(f"  {an:>12}  {r[0]:>10}  {r[1]:>10}  {r[2]:>10}")

            # ── Phase A sanity gate (full run only) ───────────────────────
            if RUN_MODE == 'full' and pn == 'A':
                ibf_a = pe['ibf']['A']['mean_cp']
                pas_a = passive_cache['A']['mean_cp']
                if ibf_a < pas_a - 10:
                    print(f"\n  ✗✗✗ PHASE A GATE FAILED: IBF ({ibf_a:+.1f}) < "
                          f"Passive ({pas_a:+.1f}) - 10 on own training context")
                    print(f"  STOP: Do not proceed to Phase B. Report telemetry.")
                    raise AssertionError(
                        f"Phase A gate: IBF {ibf_a:+.1f} < Passive {pas_a:+.1f} - 10")
                else:
                    print(f"\n  ✓ Phase A gate passed: IBF {ibf_a:+.1f} vs "
                          f"Passive {pas_a:+.1f} on Ctx A")

            # ── Checkpoints (full run only) ───────────────────────────────
            if RUN_MODE == 'full':
                ckpt_name = f"{label.replace(' ','_').lower()}_ph_{pn}.pkl"
                save_engine(eng, os.path.join(CHECKPOINT_DIR, ckpt_name))
                print(f"  Checkpoint: {ckpt_name}")
                if train_bl:
                    torch.save(mm.state_dict(), os.path.join(CHECKPOINT_DIR, f'mlp_ph_{pn}.pt'))
                    torch.save(rm.state_dict(), os.path.join(CHECKPOINT_DIR, f'replay_ph_{pn}.pt'))
                    print(f"  MLP/Replay saved: mlp_ph_{pn}.pt, replay_ph_{pn}.pt")

            eng.set_context(cid)

            # ── SMOKE RUN: exit after Phase A ─────────────────────────────
            if RUN_MODE == 'smoke' and pn == 'A':
                print(f"\n  ── SMOKE RUN COMPLETE (Phase A only) ─────────────────")
                print(f"  {eng.get_stats()}")

                # Smoke run pre-flight checks
                final_nv       = len(eng.value_centers)
                final_ncv      = sum(1 for c in eng.value_centers if c.is_crystallized())
                final_na_cryst = sum(1 for c in eng.agency_centers if c.is_crystallized())
                min_sigma      = min((c.sigma for c in eng.value_centers), default=SIGMA_TRAIN)
                checks = {
                    'centers_created':      final_nv > 50,
                    'crystallization_live':  final_ncv > 20,
                    'agency_crystallizing':  final_na_cryst > 0,
                    'sigma_shrink_started':  min_sigma < SIGMA_TRAIN * 0.95,
                    'training_unclipped':    np.array(all_R_opp_train).max() > 0.95,
                }
                print(f"\n  Pre-flight checks:")
                all_pass = True
                for k, v in checks.items():
                    print(f"    [{'✓' if v else '✗'}] {k}")
                    if not v: all_pass = False

                if all_pass:
                    print(f"\n  ✓ ALL SMOKE CHECKS PASSED")
                    print(f"    Set RUN_MODE = 'full' and rerun C8 for paper run")
                else:
                    print(f"\n  ✗ SMOKE CHECKS FAILED — diagnose before full run")

                if train_bl: return eng, pea, logs, ma, ra
                return eng, pea, logs

    finally:
        sf_train.quit()
        sf_eval_loc.quit()

    total_elapsed = time.time() - t0
    print(f"\n{'='*70}")
    print(f"  {label} COMPLETE in {total_elapsed/60:.1f} min")
    print(f"  {eng.get_stats()}")
    print(f"{'='*70}")

    if train_bl: return eng, pea, logs, ma, ra
    return eng, pea, logs


# ══════════════════════════════════════════════════════════════════════════════
#  EXECUTE TRAINING
# ══════════════════════════════════════════════════════════════════════════════

if RUN_MODE == 'smoke':
    ibf_smoke, pe_smoke, logs_smoke, mlp_smoke, replay_smoke = run_training(
        "Smoke IBF", train_bl=True)

elif RUN_MODE == 'validate':
    ibf_val, pe_val, logs_val, mlp_val, replay_val = run_training(
        "Validate IBF", train_bl=True)

    # Validation pre-flight checks
    final_nv       = len(ibf_val.value_centers)
    final_ncv      = sum(1 for c in ibf_val.value_centers if c.is_crystallized())
    final_na_cryst = sum(1 for c in ibf_val.agency_centers if c.is_crystallized())
    min_sigma      = min((c.sigma for c in ibf_val.value_centers), default=SIGMA_TRAIN)
    final_verified = sum(1 for c in ibf_val.value_centers + ibf_val.agency_centers
                         if c.crucible_verified)
    bt_a = pe_val[2]['ibf']['A']['mean_cp'] - pe_val[0]['ibf']['A']['mean_cp']
    checks = {
        'agency_crystallizing':  final_na_cryst > 0,
        'capacity_ok':           final_nv > 300,
        'sigma_shrink_active':   min_sigma < SIGMA_TRAIN * 0.85,
        'crucible_active':       final_verified > 0,
        'bt_a_positive':         bt_a > 0,
    }
    print(f"\n  {'─'*50}")
    print(f"  Pre-flight checks:")
    all_pass = True
    for k, v in checks.items():
        print(f"    [{'✓' if v else '✗'}] {k}")
        if not v: all_pass = False
    print(f"  BT_A = {bt_a:+.1f} cp")
    print(f"  {'─'*50}")
    if all_pass:
        print("  ✓ ALL CHECKS PASSED → set RUN_MODE = 'full' and rerun C8")
    else:
        print("  ✗ CHECKS FAILED — diagnose before full run")

elif RUN_MODE == 'full':
    ibf_full, pe_full, logs_full, mlp_full, replay_full = run_training(
        "Full IBF", train_bl=True)

    # ── Full performance eval ─────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"  PERFORMANCE EVAL — Full IBF")
    print(f"{'='*70}")

    eval_all = []
    for c in ['A', 'B', 'C']:
        eval_all.extend(context_eval[c][:334])

    sf_eval_c8 = get_sf_engine(threads=2, hash_mb=256)
    try:
        results_full = {}
        dr_full = DeltaROnlyAgent(ibf_full, k=K_0)
        for nm, ag in [('Full IBF',  ibf_full),
                       ('dR-only',   dr_full),
                       ('MLP',       mlp_full),
                       ('Replay',    replay_full),
                       ('Passive',   passive_agent)]:
            print(f"  Evaluating {nm}...", end='', flush=True)
            sc = eval_agent_on_fens(ag, eval_all, sf_eval_c8)
            results_full[nm] = sc
            n_clip = int(np.sum(np.abs(sc) >= CP_CLIP_EVAL))
            print(f" {np.mean(sc):+.1f} cp  (n={len(sc)}, clipped={n_clip})")

        nc = min(len(v) for v in results_full.values())
        ps = results_full['Passive'][:nc]

        print(f"\n  {'Agent':<25}  {'Mean cp':>8}  {'vs Passive':>10}  "
              f"{'W-p':>10}  {'Clip':>5}")
        print("  " + "-" * 65)
        for nm in sorted(results_full, key=lambda x: -np.mean(results_full[x][:nc])):
            a = results_full[nm][:nc]
            n_clip = int(np.sum(np.abs(a) >= CP_CLIP_EVAL))
            if nm == 'Passive':
                print(f"  {nm:<25}  {np.mean(a):>+8.1f}  {'--':>10}  "
                      f"{'--':>10}  {n_clip:>5}")
                continue
            d = a - ps
            try:    _, wp = scipy_stats.wilcoxon(d, zero_method='wilcox')
            except: wp = 1.0
            sig = '***' if wp < .001 else '**' if wp < .01 else '*' if wp < .05 else ''
            print(f"  {nm:<25}  {np.mean(a):>+8.1f}  {np.mean(d):>+10.1f}  "
                  f"{wp:.4f}{sig:>3}  {n_clip:>5}")

        print(f"\n  BT_A: {pe_full[2]['ibf']['A']['mean_cp'] - pe_full[0]['ibf']['A']['mean_cp']:+.1f} cp")
        print(f"  BT_B: {pe_full[2]['ibf']['B']['mean_cp'] - pe_full[1]['ibf']['B']['mean_cp']:+.1f} cp")

        # ── TARGET CHECK ──────────────────────────────────────────────────
        ibf_vs_pas = float(np.mean(results_full['Full IBF'][:nc] - ps))
        ibf_vs_mlp = float(np.mean(results_full['Full IBF'][:nc]) - np.mean(results_full['MLP'][:nc]))
        ibf_vs_rep = float(np.mean(results_full['Full IBF'][:nc]) - np.mean(results_full['Replay'][:nc]))
        bt_a = pe_full[2]['ibf']['A']['mean_cp'] - pe_full[0]['ibf']['A']['mean_cp']
        bt_b = pe_full[2]['ibf']['B']['mean_cp'] - pe_full[1]['ibf']['B']['mean_cp']

        print(f"\n  ── Target check ─────────────────────────────────────────────")
        targets = {
            'IBF vs Passive +70..+90':   (70, 90, ibf_vs_pas),
            'IBF beats MLP +15..+30':    (15, 30, ibf_vs_mlp),
            'IBF beats Replay +5..+20':  (5,  20, ibf_vs_rep),
            'BT_A +20..+45':             (20, 45, bt_a),
            'BT_B -5..+10':              (-5, 10, bt_b),
        }
        for desc, (lo, hi, val) in targets.items():
            hit = lo <= val <= hi
            print(f"    [{'✓' if hit else '~'}] {desc}: {val:+.1f} "
                  f"{'✓' if hit else '(outside range)'}")

        # Check: IBF must beat MLP
        if ibf_vs_mlp < 0:
            print(f"\n  ⚠ WARNING: IBF ({np.mean(results_full['Full IBF'][:nc]):+.1f}) "
                  f"is BELOW MLP ({np.mean(results_full['MLP'][:nc]):+.1f})")
            print(f"    Investigate before proceeding to C8b")

        # ── Save results with per-position persistence ────────────────────
        out_c8 = {
            'sigma': SIGMA_TRAIN, 'sigma_floor': SIGMA_FLOOR,
            'kappa': kappa_chess, 'd_eff': d_eff_12d,
            'agent_means': {nm: float(np.mean(v[:nc])) for nm, v in results_full.items()},
            'vs_passive':  {nm: float(np.mean(v[:nc] - ps))
                            for nm, v in results_full.items() if nm != 'Passive'},
            'BT_A': float(bt_a),
            'BT_B': float(bt_b),
            'per_position': {nm: v[:nc].tolist() for nm, v in results_full.items()},
            'clip_counts': {nm: int(np.sum(np.abs(v[:nc]) >= CP_CLIP_EVAL))
                            for nm, v in results_full.items()},
        }
        with open(os.path.join(CHECKPOINT_DIR, 'results_c8_full_ibf.json'), 'w') as f:
            json_mod.dump(_js(out_c8), f, indent=2)
        print(f"\n  Saved: results_c8_full_ibf.json (with per-position arrays)")

    finally:
        sf_eval_c8.quit()

  MODE: FULL RUN (120 × 100)

  Full IBF | σ=1.2693, merge=1.9040, σ_agn=0.7806
  epochs=120, games=100, eval_n=500

──────────────────────────────────────────────────────────────────────
  PHASE A — Tactical/Imbalanced (epochs 1-120)
──────────────────────────────────────────────────────────────────────
  reset_verifications() called — all crystals must re-earn broadcast
  Ep   1 (A-  1): D=0.1708, val=238c/116x/0v, agn=56c/41x, sat=0%, 85.0s
    σ: mean=1.230, min=0.587, needles=1
  Ep  10 (A- 10): D=0.1715, val=719c/623x/0v, agn=130c/121x, sat=0%, 89.6s
    σ: mean=1.078, min=0.359, needles=15
  Ep  20 (A- 20): D=0.1678, val=953c/891x/0v, agn=167c/163x, sat=0%, 93.1s
    σ: mean=1.031, min=0.332, needles=31
  Ep  30 (A- 30): D=0.1698, val=1186c/1117x/0v, agn=191c/185x, sat=0%, 92.5s
    σ: mean=1.009, min=0.332, needles=46
  Ep  40 (A- 40): D=0.1557, val=1363c/1302x/0v, agn=209c/205x, sat=0%, 96.9s
    σ: mean=0.996, min=0.332, needles=51
  Ep  50 (A- 50): D=0.1732, val=1529c/1467x/

## Note on C8 Internal Evaluation vs. C9 Paper Numbers

The performance matrix at the end of C8 evaluates IBF using **trained per-center bandwidths**, which include "needles", centers whose σ has been compressed by thermodynamic shrink. This effectively evaluates at an average scale of ~0.94× σ_train.

C9 evaluates at the **flat geometric σ* = 1.2693**, which is the paper's reported configuration (Table 2, first row). This is the "train tight, evaluate wide" principle (§5.1): corrections are learned at shrunk bandwidths for safe writing, but wider readout at σ* safely aggregates more of the verified correction landscape.

Consequently, C8 may show IBF below MLP at trained geometry, while C9 shows IBF above MLP at the geometric σ*. Both are correct, they measure different readout scales. **The paper reports C9 numbers.**

## Ablation Cascade: Isolating the Developmental Dependencies (§7.2, P6)

Three targeted ablations test the developmental cascade predicted by the theory:

| Ablation | Override | What It Removes |
|---|---|---|
| **No-Agency** | η_k = 0, w_max = 0 | Responsiveness modulation - k_eff ≡ k_0 everywhere |
| **No-Crystallization** | μ_cryst = μ_base | Stability transition - all centers decay at base rate |
| **No-Crucible** | n_cross_min = ∞ | Cross-context verification - no dissolution, no broadcast |

The predicted cascade ordering for BT_A is: Full > No-Agency > No-Crucible > No-Crystallization.

The decisive mechanistic test: without agency, the Crucible should encounter far fewer contradiction zones, collapsing dissolution events from thousands to near-zero. This proves that agency is not merely a temperature schedule — it is the developmental driver that feeds the self-correction mechanism.

In [12]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Cell C8b TRAINING: Ablations (No-Agency, No-Cryst, No-Crucible)         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ══════════════════════════════════════════════════════════════════════════════
# Requires C8 to have completed with RUN_MODE='full'
# (ibf_full, pe_full, mlp_full, replay_full in memory).
#
# Uses run_training from C8 (which uses stockfish_eval_cp_raw for training
# and stockfish_eval_cp_clipped for evaluation).
# ══════════════════════════════════════════════════════════════════════════════

assert RUN_MODE == 'full', (
    f"C8b requires RUN_MODE='full' (got '{RUN_MODE}'). "
    "Run C8 in full mode first.")
assert 'ibf_full' in dir(), "ibf_full not found — run C8 in full mode first"

def run_or_load_ablation(label, ov):
    """Run ablation or load from checkpoint if Phase C exists."""
    ckpt = os.path.join(CHECKPOINT_DIR,
                        f"{label.replace(' ','_').lower()}_ph_C.pkl")
    if os.path.exists(ckpt):
        print(f"\n  [{label}] Phase C checkpoint found — loading")
        p = build_params(**ov)
        eng = load_engine(ckpt, p)
        # Reconstruct pe from saved phase checkpoints
        pe = []
        sf_r = get_sf_engine(threads=1, hash_mb=128)
        try:
            for pn in ['A', 'B', 'C']:
                ph_ckpt = os.path.join(CHECKPOINT_DIR,
                    f"{label.replace(' ','_').lower()}_ph_{pn}.pkl")
                eng_ph = load_engine(ph_ckpt, p)
                pe_ph = {'phase': pn, 'ibf': {}, 'passive': passive_cache}
                for ctx in ['A', 'B', 'C']:
                    eng_ph.set_context({'A': 0, 'B': 1, 'C': 2}[ctx])
                    pe_ph['ibf'][ctx] = eval_agent_on_context(
                        eng_ph, ctx, sf_r, n=N_EVAL_TRAINING)
                pe.append(pe_ph)
                print(f"    Phase {pn}: A={pe_ph['ibf']['A']['mean_cp']:+.1f}, "
                      f"B={pe_ph['ibf']['B']['mean_cp']:+.1f}, "
                      f"C={pe_ph['ibf']['C']['mean_cp']:+.1f}")
        finally:
            sf_r.quit()
        print(f"  [{label}] Loaded: {eng.get_stats()}")
        return eng, pe, []
    else:
        return run_training(label, ov=ov)

# ── Run ablations ──────────────────────────────────────────────────────────────
ibf_noagn,   pe_noagn,   logs_noagn   = run_or_load_ablation(
    "No-Agency",   {'eta_k': 0.0, 'w_max': 0.0})

ibf_nocryst, pe_nocryst, logs_nocryst = run_or_load_ablation(
    "No-Cryst",    {'mu_crystallized': MU_BASE})

ibf_nocruc,  pe_nocruc,  logs_nocruc  = run_or_load_ablation(
    "No-Crucible", {'n_cross_min': 999999})

# ══════════════════════════════════════════════════════════════════════════════
#  FULL PERFORMANCE EVAL — All Configurations
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"  FULL PERFORMANCE EVAL — All Configurations")
print(f"{'='*70}")

# eval_all from C8; rebuild if needed
if 'eval_all' not in globals() or len(eval_all) == 0:
    eval_all = []
    for c in ['A', 'B', 'C']:
        eval_all.extend(context_eval[c][:334])

sf_eval_c8b = get_sf_engine(threads=2, hash_mb=256)

try:
    results_all = {}

    agents_to_eval = [
        ('Full IBF',    ibf_full),
        ('No-Agency',   ibf_noagn),
        ('No-Cryst',    ibf_nocryst),
        ('No-Crucible', ibf_nocruc),
        ('MLP',         mlp_full),
        ('Replay',      replay_full),
        ('Passive',     passive_agent),
    ]

    for nm, ag in agents_to_eval:
        print(f"  Evaluating {nm}...", end='', flush=True)
        sc = eval_agent_on_fens(ag, eval_all, sf_eval_c8b)
        results_all[nm] = sc
        n_clip = int(np.sum(np.abs(sc) >= CP_CLIP_EVAL))
        print(f" {np.mean(sc):+.1f} cp  (n={len(sc)}, clipped={n_clip})")

    nc = min(len(v) for v in results_all.values())
    ps = results_all['Passive'][:nc]

    # ── Main comparison table ─────────────────────────────────────────────────
    print(f"\n  {'Agent':<25}  {'Mean cp':>8}  {'vs Passive':>10}  "
          f"{'W-p':>10}  {'Clip':>5}")
    print("  " + "-" * 65)
    for nm in sorted(results_all, key=lambda x: -np.mean(results_all[x][:nc])):
        a = results_all[nm][:nc]
        n_clip = int(np.sum(np.abs(a) >= CP_CLIP_EVAL))
        if nm == 'Passive':
            print(f"  {nm:<25}  {np.mean(a):>+8.1f}  {'--':>10}  "
                  f"{'--':>10}  {n_clip:>5}")
            continue
        d = a - ps
        try:    _, wp = scipy_stats.wilcoxon(d, zero_method='wilcox')
        except: wp = 1.0
        sig = '***' if wp < .001 else '**' if wp < .01 else '*' if wp < .05 else ''
        print(f"  {nm:<25}  {np.mean(a):>+8.1f}  {np.mean(d):>+10.1f}  "
              f"{wp:.4f}{sig:>3}  {n_clip:>5}")

    # ── Backward Transfer ─────────────────────────────────────────────────────
    def gcp(pe, pi, ag, ctx): return pe[pi][ag][ctx]['mean_cp']
    def BT_A(pe, ag): return gcp(pe, 2, ag, 'A') - gcp(pe, 0, ag, 'A')
    def BT_B(pe, ag): return gcp(pe, 2, ag, 'B') - gcp(pe, 1, ag, 'B')

    print(f"\n  --- Backward Transfer ---")
    print(f"  {'Agent':<25}  {'BT_A':>8}  {'BT_B':>8}")
    for lbl, pe in [('Full IBF',    pe_full),
                    ('No-Agency',   pe_noagn),
                    ('No-Cryst',    pe_nocryst),
                    ('No-Crucible', pe_nocruc)]:
        print(f"  {lbl:<25}  {BT_A(pe,'ibf'):>+8.1f}  {BT_B(pe,'ibf'):>+8.1f}")
    for lbl in ['mlp', 'replay']:
        print(f"  {lbl.upper():<25}  {BT_A(pe_full,lbl):>+8.1f}  {BT_B(pe_full,lbl):>+8.1f}")

    # ── Cascade (BT_A) ────────────────────────────────────────────────────────
    print(f"\n  --- Cascade (BT_A) ---")
    cascade = {
        'Full':     BT_A(pe_full,    'ibf'),
        'No-Agn':   BT_A(pe_noagn,  'ibf'),
        'No-Cruc':  BT_A(pe_nocruc, 'ibf'),
        'No-Cryst': BT_A(pe_nocryst,'ibf'),
    }
    sc_sorted = sorted(cascade.items(), key=lambda x: -x[1])
    print("  Predicted: Full > No-Agn > No-Cruc > No-Cryst")
    print("  Actual:    " + " > ".join(f"{k}({v:+.1f})" for k, v in sc_sorted))

    # ── Crucible stats ────────────────────────────────────────────────────────
    print(f"\n  --- Crucible ---")
    for lbl, eng in [('Full',     ibf_full),
                     ('No-Agn',   ibf_noagn),
                     ('No-Cryst', ibf_nocryst)]:
        nd = sum(len(c.dissolution_log)
                 for c in eng.value_centers + eng.agency_centers)
        nv = sum(1 for c in eng.value_centers + eng.agency_centers
                 if c.crucible_verified)
        ne = sum(1 for c in eng.value_centers + eng.agency_centers
                 if c.was_ever_crystallized)
        print(f"  {lbl:<12}: {ne} ever-cryst, {nv} verified, {nd} dissolved")

    # ── Target checks ─────────────────────────────────────────────────────────
    ibf_vs_pas = float(np.mean(results_all['Full IBF'][:nc] - ps))
    noagn_vs_pas = float(np.mean(results_all['No-Agency'][:nc] - ps))
    total_dissolutions = sum(len(c.dissolution_log)
                             for c in ibf_full.value_centers + ibf_full.agency_centers)

    print(f"\n  ── Target check ─────────────────────────────────────────────")
    targets = {
        'No-Agency vs Passive +30..+50': (30, 50, noagn_vs_pas),
        'Dissolutions 1000..5000':       (1000, 5000, total_dissolutions),
    }
    for desc, (lo, hi, val) in targets.items():
        hit = lo <= val <= hi
        vstr = f"{val:+.1f}" if isinstance(val, float) else f"{val}"
        print(f"    [{'✓' if hit else '~'}] {desc}: {vstr} "
              f"{'✓' if hit else '(outside range)'}")

    # ── Save combined results with per-position persistence ───────────────────
    out_c8b = {
        'sigma': SIGMA_TRAIN, 'sigma_floor': SIGMA_FLOOR,
        'kappa': kappa_chess, 'd_eff': d_eff_12d,
        'agent_means': {nm: float(np.mean(v[:nc])) for nm, v in results_all.items()},
        'vs_passive':  {nm: float(np.mean(v[:nc] - ps))
                        for nm, v in results_all.items() if nm != 'Passive'},
        'cascade': cascade,
        'BT': {lbl: {'BT_A': float(BT_A(pe, 'ibf')), 'BT_B': float(BT_B(pe, 'ibf'))}
               for lbl, pe in [('Full IBF',    pe_full),
                                ('No-Agency',   pe_noagn),
                                ('No-Cryst',    pe_nocryst),
                                ('No-Crucible', pe_nocruc)]},
        'per_position': {nm: v[:nc].tolist() for nm, v in results_all.items()},
        'clip_counts': {nm: int(np.sum(np.abs(v[:nc]) >= CP_CLIP_EVAL))
                        for nm, v in results_all.items()},
        'crucible': {
            lbl: {
                'ever_crystallized': sum(1 for c in eng.value_centers + eng.agency_centers
                                         if c.was_ever_crystallized),
                'verified': sum(1 for c in eng.value_centers + eng.agency_centers
                                if c.crucible_verified),
                'dissolved': sum(len(c.dissolution_log)
                                 for c in eng.value_centers + eng.agency_centers),
            }
            for lbl, eng in [('Full IBF', ibf_full), ('No-Agency', ibf_noagn),
                              ('No-Cryst', ibf_nocryst), ('No-Crucible', ibf_nocruc)]
        },
    }
    with open(os.path.join(CHECKPOINT_DIR, 'results_c8b_all.json'), 'w') as f:
        json_mod.dump(_js(out_c8b), f, indent=2)
    print(f"\n  Saved: results_c8b_all.json (with per-position arrays)")

finally:
    sf_eval_c8b.quit()


  No-Agency | σ=1.2693, merge=1.9040, σ_agn=0.7806
  Overrides: {'eta_k': 0.0, 'w_max': 0.0}
  epochs=120, games=100, eval_n=500

──────────────────────────────────────────────────────────────────────
  PHASE A — Tactical/Imbalanced (epochs 1-120)
──────────────────────────────────────────────────────────────────────
  reset_verifications() called — all crystals must re-earn broadcast
  Ep   1 (A-  1): D=0.1694, val=249c/125x/0v, agn=56c/44x, sat=0%, 92.0s
    σ: mean=1.229, min=0.576, needles=1
  Ep  10 (A- 10): D=0.1732, val=674c/600x/0v, agn=123c/115x, sat=0%, 92.5s
    σ: mean=1.078, min=0.373, needles=19
  Ep  20 (A- 20): D=0.1662, val=932c/856x/0v, agn=164c/156x, sat=0%, 93.1s
    σ: mean=1.035, min=0.373, needles=29
  Ep  30 (A- 30): D=0.1699, val=1129c/1072x/0v, agn=191c/185x, sat=0%, 100.1s
    σ: mean=1.012, min=0.368, needles=36
  Ep  40 (A- 40): D=0.1640, val=1324c/1249x/0v, agn=215c/209x, sat=0%, 101.4s
    σ: mean=0.998, min=0.330, needles=50
  Ep  50 (A- 50): D=0.1638, 

## Paper Table 2: Final Evaluation (§7.2)

C8 training-time evaluation is contaminated by ALPHA_SHRINK = 500, which compresses
77% of center bandwidths below prescribed σ during training trough a thermodynamic shrink mechanism. C13 ablation confirms this accounts for the entire performance gap (23.4 cp) between trained geometry (+65.6 cp, below MLP) and prescribed σ (+89.6 cp, above MLP). Thermodynamic shrink is a resolution artifact, not a mechanism. The Crucible handles quarantine via dissolution (18,986 events). Shrink will be removed in the next iteration.

**The true performance numbers come from this cell (C9) and C12**, which evaluate
all agents at the geometrically prescribed σ* = 1.2693. Three seeds confirm:
+88.9 ± 2.8 cp vs passive, exceeding both MLP (+82.3) and Replay (+76.4).

This cell also produces: backward transfer, cascade ordering, Crucible statistics,
agency analysis (k_eff, partial correlation), and σ-shrink diagnostics (retained
for documentation, no longer part of the paper narrative).

All evaluation uses Stockfish depth 8, deterministic greedy, ±1000 cp clip, n = 1,002.

In [13]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Cell C9: EVALUATION                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ══════════════════════════════════════════════════════════════════════════════
# IMPORTANT: Run C9 BEFORE C10.
# C10 captures orig_sigmas and restores them after the sweep, but running
# C9 first captures the real trained geometry including needles.
#
# All evaluation uses stockfish_eval_cp_clipped (±1000 cp).
# Per-position score arrays saved in paper_results.json.
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70 + "\n  C9 — EVALUATION\n" + "=" * 70)

eval_all = []
for c in ['A', 'B', 'C']:
    eval_all.extend(context_eval[c][:334])

# Capture trained sigmas BEFORE any flat overrides — needed to restore
# needle geometry for the σ-shrink verification later in this cell.
orig_sigmas_c9      = [c.sigma for c in ibf_full.value_centers]
orig_sigmas_c9_noag = [c.sigma for c in ibf_noagn.value_centers]

def set_sigma_flat(eng, s):
    """Flat override — used ONLY for the s=1.0 comparison row, not for sweep."""
    for c in eng.value_centers: c.sigma = s

def restore_trained_sigmas():
    """Restore exact trained geometry including needles."""
    for c, s in zip(ibf_full.value_centers, orig_sigmas_c9):
        c.sigma = s
    for c, s in zip(ibf_noagn.value_centers, orig_sigmas_c9_noag):
        c.sigma = s

sf_c9 = get_sf_engine(threads=2, hash_mb=256)

try:
    results = {}

    # ── Eval at two flat σ scales (for comparison table) ──────────────────────
    for sl, sv in [('s=1.0', 1.0), (f's={SIGMA_TRAIN:.2f}', SIGMA_TRAIN)]:
        set_sigma_flat(ibf_full, sv)
        set_sigma_flat(ibf_noagn, sv)
        dr = DeltaROnlyAgent(ibf_full, k=K_0)
        for nm, ag in [(f'Full IBF {sl}',  ibf_full),
                       (f'No-Agency {sl}',  ibf_noagn),
                       (f'dR-only {sl}',    dr)]:
            results[nm] = eval_agent_on_fens(ag, eval_all, sf_c9)

    # Restore exact trained geometry (including needles) before proceeding.
    # DO NOT use set_sigma_flat here — it would overwrite shrunk centers.
    restore_trained_sigmas()

    results['MLP']     = eval_agent_on_fens(mlp_full,      eval_all, sf_c9)
    results['Replay']  = eval_agent_on_fens(replay_full,   eval_all, sf_c9)
    results['Passive'] = eval_agent_on_fens(passive_agent, eval_all, sf_c9)

    nc = min(len(v) for v in results.values())
    ps = results['Passive'][:nc]

    print(f"\n  {'Agent':<30}  {'Mean':>8}  {'vPass':>8}  {'W-p':>10}  {'Clip':>5}")
    print("  " + "-" * 67)
    for nm in sorted(results, key=lambda x: -np.mean(results[x][:nc])):
        a = results[nm][:nc]
        n_clip = int(np.sum(np.abs(a) >= CP_CLIP_EVAL))
        if nm == 'Passive':
            print(f"  {nm:<30}  {np.mean(a):>+8.1f}  {'--':>8}  {'--':>10}  {n_clip:>5}")
            continue
        d = a - ps
        try:
            _, wp = scipy_stats.wilcoxon(d, zero_method='wilcox')
        except:
            wp = 1.0
        sig = '***' if wp < .001 else '**' if wp < .01 else '*' if wp < .05 else ''
        print(f"  {nm:<30}  {np.mean(a):>+8.1f}  {np.mean(d):>+8.1f}  "
              f"{wp:.4f}{sig:>3}  {n_clip:>5}")

    # ── Backward Transfer ─────────────────────────────────────────────────────
    def gcp(pe, pi, ag, ctx): return pe[pi][ag][ctx]['mean_cp']
    def BT_A(pe, ag): return gcp(pe, 2, ag, 'A') - gcp(pe, 0, ag, 'A')
    def BT_B(pe, ag): return gcp(pe, 2, ag, 'B') - gcp(pe, 1, ag, 'B')

    print(f"\n  --- Backward Transfer ---")
    print(f"  {'Agent':<25}  {'BT_A':>8}  {'BT_B':>8}")
    for lbl, pe in [('Full IBF',    pe_full),
                    ('No-Agency',   pe_noagn),
                    ('No-Cryst',    pe_nocryst),
                    ('No-Crucible', pe_nocruc)]:
        print(f"  {lbl:<25}  {BT_A(pe,'ibf'):>+8.1f}  {BT_B(pe,'ibf'):>+8.1f}")
    for lbl in ['mlp', 'replay']:
        print(f"  {lbl.upper():<25}  {BT_A(pe_full,lbl):>+8.1f}  {BT_B(pe_full,lbl):>+8.1f}")

    # ── Cascade (BT_A) ────────────────────────────────────────────────────────
    print(f"\n  --- Cascade (BT_A) ---")
    cascade = {
        'Full':     BT_A(pe_full,    'ibf'),
        'No-Agn':   BT_A(pe_noagn,  'ibf'),
        'No-Cruc':  BT_A(pe_nocruc, 'ibf'),
        'No-Cryst': BT_A(pe_nocryst,'ibf'),
    }
    sc_sorted = sorted(cascade.items(), key=lambda x: -x[1])
    print("  Predicted: Full > No-Agn > No-Cruc > No-Cryst")
    print("  Actual:    " + " > ".join(f"{k}({v:+.1f})" for k, v in sc_sorted))

    # ── Crucible stats ────────────────────────────────────────────────────────
    print(f"\n  --- Crucible ---")
    for lbl, eng in [('Full', ibf_full), ('No-Agn', ibf_noagn)]:
        nd  = sum(len(c.dissolution_log)
                  for c in eng.value_centers + eng.agency_centers)
        nv  = sum(1 for c in eng.value_centers + eng.agency_centers
                  if c.crucible_verified)
        ne  = sum(1 for c in eng.value_centers + eng.agency_centers
                  if c.was_ever_crystallized)
        print(f"  {lbl}: {ne} ever-cryst, {nv} verified, {nd} dissolved")

    # ── Agency analysis ───────────────────────────────────────────────────────
    print(f"\n  --- Agency (k_eff vs entropy) ---")
    kv, hv, nv_ = [], [], []
    for fen in eval_all:
        board = chess.Board(fen)
        if board.is_game_over(): continue
        nl = len(list(board.legal_moves))
        if nl < 2: continue
        ctx = classify_context(board)
        ibf_full.set_context({'A': 0, 'B': 1, 'C': 2}.get(ctx, 0) if ctx else 0)
        z = RC_encode(board)
        k = ibf_full.k_eff(z)
        _, dg = ibf_full.select_move(board, deterministic=False)
        H = dg.get('entropy', 0.0)
        Hn = H / np.log(nl) if nl > 1 else 0.0
        kv.append(k); hv.append(Hn); nv_.append(nl)
    kv, hv, nv_ = np.array(kv), np.array(hv), np.array(nv_)
    if np.std(kv) > 0.01:
        rkH, _ = scipy_stats.spearmanr(kv, hv)
        rkN, _ = scipy_stats.spearmanr(kv, nv_)
        rHN, _ = scipy_stats.spearmanr(hv, nv_)
        den = np.sqrt((1 - rkN**2) * (1 - rHN**2))
        rp  = (rkH - rkN * rHN) / den if den > 1e-6 else 0.0
        ne2 = len(kv) - 1
        zf  = 0.5 * np.log((1 + rp) / (1 - rp + 1e-10))
        se  = 1.0 / np.sqrt(ne2 - 3) if ne2 > 3 else 1e6
        pp  = 2 * (1 - scipy_stats.norm.cdf(abs(zf / se)))
        print(f"  rho_partial={rp:.3f}, p={pp:.4f}")
        print(f"  k_eff: mean={np.mean(kv):.2f}, std={np.std(kv):.2f}, "
              f"range=[{np.min(kv):.2f}, {np.max(kv):.2f}]")
    else:
        print(f"  k_eff has no variance (std={np.std(kv):.4f}) — "
              f"agency channel may not have crystallized")
        print(f"  Agency centers: {len(ibf_full.agency_centers)}, "
              f"crystallized: {sum(1 for c in ibf_full.agency_centers if c.is_crystallized())}")

    # ── Physicist's prediction: σ shrink verification ─────────────────────────
    # MUST run before C10 — C10 modifies sigmas during sweep
    print(f"\n  --- Thermodynamic σ-shrink verification ---")
    print(f"  Prediction: boundary crystals compress from "
          f"σ_train={SIGMA_TRAIN:.3f} toward σ_floor={SIGMA_FLOOR:.3f}")
    all_vc_sigs = np.array([c.sigma for c in ibf_full.value_centers])
    cryst_sigs  = np.array([c.sigma for c in ibf_full.value_centers
                            if c.is_crystallized()])
    shrunk_sigs = cryst_sigs[cryst_sigs < SIGMA_TRAIN * 0.85] if len(cryst_sigs) else np.array([])
    print(f"  All value centers:     n={len(all_vc_sigs)}, "
          f"mean σ={np.mean(all_vc_sigs):.4f}, min={np.min(all_vc_sigs):.4f}")
    if len(cryst_sigs):
        print(f"  Crystallized centers:  n={len(cryst_sigs)}, "
              f"mean σ={np.mean(cryst_sigs):.4f}")
    else:
        print(f"  Crystallized centers:  n=0")
    if len(shrunk_sigs):
        print(f"  Shrunk (<0.85×σ_train): n={len(shrunk_sigs)}, "
              f"mean σ={np.mean(shrunk_sigs):.4f}")
        bleed_shrunk = np.exp(-3.04**2 / (2 * np.mean(shrunk_sigs)**2))
        print(f"  Mean bleed at shrunk σ across d(B,C)=3.04: "
              f"{bleed_shrunk*100:.8f}%")
        print(f"  (cf. {100*np.exp(-3.04**2/(2*SIGMA_TRAIN**2)):.3f}% at σ_train, "
              f"{100*np.exp(-3.04**2/(2*SIGMA_FLOOR**2)):.8f}% at σ_floor)")
    else:
        print(f"  Shrunk (<0.85×σ_train): n=0")

    # ── Save results ──────────────────────────────────────────────────────────
    out = {
        'sigma': SIGMA_TRAIN, 'sigma_floor': SIGMA_FLOOR,
        'kappa': kappa_chess,  'd_eff': d_eff_12d,
        'cascade': cascade,
        'BT_A_full': BT_A(pe_full, 'ibf'),
        'BT_B_full': BT_B(pe_full, 'ibf'),
        'agent_means': {nm: float(np.mean(v[:nc])) for nm, v in results.items()},
        'vs_passive':  {nm: float(np.mean(v[:nc] - ps))
                        for nm, v in results.items() if nm != 'Passive'},
        'per_position': {nm: v[:nc].tolist() for nm, v in results.items()},
        'clip_counts': {nm: int(np.sum(np.abs(v[:nc]) >= CP_CLIP_EVAL))
                        for nm, v in results.items()},
        'shrink_n': int(len(shrunk_sigs)),
        'shrink_mean_sigma': float(np.mean(shrunk_sigs)) if len(shrunk_sigs) else None,
        'agency': {
            'k_eff_mean': float(np.mean(kv)) if len(kv) else None,
            'k_eff_std':  float(np.std(kv))  if len(kv) else None,
            'rho_partial': float(rp) if np.std(kv) > 0.01 else None,
            'rho_partial_p': float(pp) if np.std(kv) > 0.01 else None,
        },
    }
    with open(os.path.join(CHECKPOINT_DIR, 'paper_results.json'), 'w') as f:
        json_mod.dump(_js(out), f, indent=2)
    print(f"\n  Results saved: {CHECKPOINT_DIR}paper_results.json")

finally:
    sf_c9.quit()

print(f"\n{'='*70}\n  C9 COMPLETE\n{'='*70}")

  C9 — EVALUATION

  Agent                               Mean     vPass         W-p   Clip
  -------------------------------------------------------------------
  dR-only s=1.27                    -241.2     +92.0  0.0000***     14
  Full IBF s=1.27                   -243.0     +90.2  0.0000***     14
  dR-only s=1.0                     -249.6     +83.6  0.0000***     16
  Full IBF s=1.0                    -250.5     +82.7  0.0000***     15
  MLP                               -250.9     +82.3  0.0000***     12
  No-Agency s=1.27                  -251.2     +81.9  0.0000***     14
  Replay                            -256.8     +76.4  0.0000***     15
  No-Agency s=1.0                   -265.3     +67.9  0.0000***     18
  Passive                           -333.2        --          --     13

  --- Backward Transfer ---
  Agent                          BT_A      BT_B
  Full IBF                      +38.5      +0.4
  No-Agency                     +16.0      -3.4
  No-Cryst                

## Train Tight, Evaluate Wide - Post-Training σ Sweep (§5.1, §7.2)

After training at the geometrically calibrated bandwidth σ*, the system can be evaluated across a range of σ_eval values **without retraining**. This sweep tests the paper's claim that write geometry and read geometry need not coincide.

In chess, the Crucible explains the difference (§5.1): *"once cross-context verification has confirmed that surviving corrections are mutually consistent, wider aggregation can safely aggregate more verified corrections per query without introducing contradiction."*

The sweep scales each center's σ from its **trained value** (preserving needle geometry), not from a flat σ_train. Scale = 1.0 corresponds to the exact trained geometry.

Expected: monotonic rise from scale 0.7 to peak around scale 1.3, then broad plateau through ~1.8, then gradual decline.

In [14]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  # C10: σ SWEEP                                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ══════════════════════════════════════════════════════════════════════════════
# Sweeps σ_eval from scale 0.7 to 4.5 using set_sigma_relative —
# scales each center from its TRAINED sigma, not flat from σ_train.
# A needle at 0.136 evaluates at 0.177 at scale=1.3, not 1.33.
# scale=1.0 = exact trained geometry.
#
# All evaluation uses stockfish_eval_cp_clipped (±1000 cp).
# Captures orig_sigmas before sweep, restores exact trained geometry after.
#
# IMPORTANT: Run C9 BEFORE C10. C9 captures σ-shrink stats from trained
# geometry. This cell modifies and restores sigmas during the sweep.
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70 + "\n  C10 — σ SWEEP\n" + "=" * 70)
print(f"  Sweeping around σ_train={SIGMA_TRAIN:.4f} (P10/3.0 generalization anchor)")
print(f"  κ_chess={kappa_chess:.4f}, d_eff={d_eff_12d:.1f}")
print(f"  Note: scale applied per-center from trained σ, not flat from σ_train")

# Store original trained sigmas ONCE before sweep
orig_sigmas = [c.sigma for c in ibf_full.value_centers]

def set_sigma_relative(eng, scale, orig_sigs):
    """Scale each center's σ from its trained value. Preserves needle geometry."""
    for c, orig_s in zip(eng.value_centers, orig_sigs):
        c.sigma = orig_s * scale

sf_c10 = get_sf_engine(threads=2, hash_mb=256)
sweep = {}

try:
    for scale in [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5,
                  1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2, 2.3, 2.4,
                  2.5, 2.6, 2.7, 3.0, 3.5, 4.0, 4.5]:
        sv = SIGMA_TRAIN * scale  # for display only
        set_sigma_relative(ibf_full, scale, orig_sigmas)

        si, sp = [], []
        for fen in eval_all:
            board = chess.Board(fen)
            if board.is_game_over(): continue
            ctx = classify_context(board)
            ibf_full.set_context({'A': 0, 'B': 1, 'C': 2}.get(ctx, 0) if ctx else 0)
            mi, _ = ibf_full.select_move(board, deterministic=True)
            mp, _ = passive_agent.select_move(board, deterministic=True)
            if mi and mp:
                si.append(-stockfish_eval_cp_clipped(apply_move(board, mi), sf_c10))
                sp.append(-stockfish_eval_cp_clipped(apply_move(board, mp), sf_c10))

        d = np.array(si) - np.array(sp)
        n_clip_ibf = int(np.sum(np.abs(np.array(si)) >= CP_CLIP_EVAL))
        try:
            _, wp = scipy_stats.wilcoxon(d, zero_method='wilcox')
        except:
            wp = 1.0
        sweep[scale] = {
            'delta': float(np.mean(d)),
            'p': float(wp),
            'sigma_abs': float(sv),
            'n_clip': n_clip_ibf,
        }
        marker = ' ← trained geometry' if abs(scale - 1.0) < 0.05 else ''
        print(f"  scale={scale:.1f}  σ={sv:.4f}  {np.mean(d):>+7.1f} cp  "
              f"p={wp:.4f}  clip={n_clip_ibf}{marker}")

finally:
    sf_c10.quit()

# Restore exact trained geometry
for c, orig_s in zip(ibf_full.value_centers, orig_sigmas):
    c.sigma = orig_s

# ── Analysis ──────────────────────────────────────────────────────────────────
peak = max(sweep, key=lambda s: sweep[s]['delta'])
print(f"\n  Peak: scale={peak:.1f} (σ={sweep[peak]['sigma_abs']:.4f}, "
      f"{sweep[peak]['delta']:+.1f} cp)")

if peak == 1.0:
    print("  ★ Material constant CONFIRMED — κ-derived σ is optimal")
elif abs(peak - 1.0) <= 0.1:
    print(f"  ✓ Peak within 10% of κ-derived σ — consistent with theory")
elif 1.0 < peak <= 2.0:
    print(f"  ✓ Peak at scale={peak:.1f} — train-tight-evaluate-wide regime")
    print(f"    Corrections learned at training σ combine more fully at wider readout")
elif peak > 2.0:
    print(f"  NOTE: Peak at scale={peak:.1f} — wider than expected. "
          f"Report honestly.")
else:
    print(f"  NOTE: Peak at scale={peak:.1f} (<1.0) — narrower than training σ. "
          f"Report honestly.")

# Target check
if 1.0 <= peak <= 2.0:
    print(f"  ✓ σ sweep peak in target range [1.0, 2.0]")
else:
    print(f"  ~ σ sweep peak {peak:.1f} outside target range [1.0, 2.0]")

print(f"\n  σ_train={SIGMA_TRAIN:.4f} (P10/3), κ={kappa_chess:.4f}, d_eff={d_eff_12d:.1f}")
print(f"  σ_floor={SIGMA_FLOOR:.4f} (sib/20)")

# ── Append sweep results to paper_results.json ────────────────────────────────
with open(os.path.join(CHECKPOINT_DIR, 'paper_results.json'), 'r') as f:
    existing = json_mod.load(f)

existing['sweep'] = _js(sweep)
existing['peak_scale'] = float(peak)
existing['peak_sigma'] = float(sweep[peak]['sigma_abs'])
existing['peak_delta_cp'] = float(sweep[peak]['delta'])

with open(os.path.join(CHECKPOINT_DIR, 'paper_results.json'), 'w') as f:
    json_mod.dump(existing, f, indent=2)

print(f"\n  Sweep results appended to paper_results.json")
print(f"\n{'='*70}\n  C10 COMPLETE — {CHECKPOINT_DIR}paper_results.json\n{'='*70}")

  C10 — σ SWEEP
  Sweeping around σ_train=1.2693 (P10/3.0 generalization anchor)
  κ_chess=0.5747, d_eff=4.9
  Note: scale applied per-center from trained σ, not flat from σ_train
  scale=0.7  σ=0.8885    +17.6 cp  p=0.0227  clip=13
  scale=0.8  σ=1.0155    +33.2 cp  p=0.0000  clip=14
  scale=0.9  σ=1.1424    +52.0 cp  p=0.0000  clip=15
  scale=1.0  σ=1.2693    +65.6 cp  p=0.0000  clip=14 ← trained geometry
  scale=1.1  σ=1.3963    +75.7 cp  p=0.0000  clip=14
  scale=1.2  σ=1.5232    +82.7 cp  p=0.0000  clip=16
  scale=1.3  σ=1.6501    +94.6 cp  p=0.0000  clip=16
  scale=1.4  σ=1.7771    +87.2 cp  p=0.0000  clip=15
  scale=1.5  σ=1.9040    +86.3 cp  p=0.0000  clip=15
  scale=1.6  σ=2.0309    +85.8 cp  p=0.0000  clip=15
  scale=1.7  σ=2.1579    +85.0 cp  p=0.0000  clip=16
  scale=1.8  σ=2.2848    +83.9 cp  p=0.0000  clip=18
  scale=1.9  σ=2.4117    +81.8 cp  p=0.0000  clip=19
  scale=2.0  σ=2.5387    +80.3 cp  p=0.0000  clip=18
  scale=2.1  σ=2.6656    +71.9 cp  p=0.0000  clip=19
  scal

## Extra Seeds (C11)
Runs seeds 43 and 44 using the same frozen encoder, σ calibration, context pools,
and passive baseline as the main run (seed 42). Only training randomness differs.
Combined with seed 42, produces three-seed summary:
  BT_A = +35.4 ± 2.9 cp | vs passive = +61.8 ± 1.9 cp (trained geometry)
  At prescribed σ_train (C12): +88.9 ± 2.8 cp, all seeds exceed MLP and Replay.

In [15]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  # C11: EXTRA SEEDS                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ══════════════════════════════════════════════════════════════════════════════

SEEDS_EXTRA = [43, 44]

seed_results = {}

for seed_i in SEEDS_EXTRA:
    print(f"\n{'#'*70}")
    print(f"  SEED {seed_i}")
    print(f"{'#'*70}")

    # Reset randomness — encoder and σ values are frozen, only training changes
    random.seed(seed_i)
    np.random.seed(seed_i)
    torch.manual_seed(seed_i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_i)

    # Run full training (no baselines — MLP/Replay not needed for seed variance)
    eng_s, pe_s, logs_s = run_training(
        f"Full IBF Seed {seed_i}", train_bl=False)

    # ── Final eval on shared eval pool ────────────────────────────────────────
    eval_all_s = []
    for c in ['A', 'B', 'C']:
        eval_all_s.extend(context_eval[c][:334])

    sf_s = get_sf_engine(threads=2, hash_mb=256)
    try:
        sc_ibf = eval_agent_on_fens(eng_s, eval_all_s, sf_s)
        sc_pas = eval_agent_on_fens(passive_agent, eval_all_s, sf_s)
    finally:
        sf_s.quit()

    nc_s = min(len(sc_ibf), len(sc_pas))
    d_s = sc_ibf[:nc_s] - sc_pas[:nc_s]
    try:    _, wp_s = scipy_stats.wilcoxon(d_s, zero_method='wilcox')
    except: wp_s = 1.0

    bt_a = pe_s[2]['ibf']['A']['mean_cp'] - pe_s[0]['ibf']['A']['mean_cp']
    bt_b = pe_s[2]['ibf']['B']['mean_cp'] - pe_s[1]['ibf']['B']['mean_cp']
    vs_pas = float(np.mean(d_s))

    print(f"\n  Seed {seed_i} results:")
    print(f"    IBF vs Passive: {vs_pas:+.1f} cp  (p={wp_s:.4f})")
    print(f"    BT_A: {bt_a:+.1f} cp")
    print(f"    BT_B: {bt_b:+.1f} cp")

    # Crucible stats
    nd = sum(len(c.dissolution_log)
             for c in eng_s.value_centers + eng_s.agency_centers)
    nv = sum(1 for c in eng_s.value_centers + eng_s.agency_centers
             if c.crucible_verified)
    print(f"    Crucible: {nv} verified, {nd} dissolved")

    seed_results[seed_i] = {
        'vs_passive': vs_pas,
        'p_value': float(wp_s),
        'BT_A': float(bt_a),
        'BT_B': float(bt_b),
        'ibf_mean': float(np.mean(sc_ibf[:nc_s])),
        'passive_mean': float(np.mean(sc_pas[:nc_s])),
        'n_verified': nv,
        'n_dissolved': nd,
        'per_position_ibf': sc_ibf[:nc_s].tolist(),
        'per_position_passive': sc_pas[:nc_s].tolist(),
        'phase_evals': {
            pn: {ctx: pe_s[pi]['ibf'][ctx]['mean_cp']
                 for ctx in ['A', 'B', 'C']}
            for pi, pn in enumerate(['A', 'B', 'C'])
        },
    }

    # Save checkpoint per seed
    save_engine(eng_s, os.path.join(CHECKPOINT_DIR, f'full_ibf_seed_{seed_i}.pkl'))

    # Free memory
    del eng_s, pe_s, logs_s
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Cross-seed summary ────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  CROSS-SEED SUMMARY")
print(f"{'='*70}")

# Include seed 42 (main run)
all_seeds = {42: {
    'vs_passive': float(np.mean(results_full['Full IBF'][:nc] - results_full['Passive'][:nc])),
    'BT_A': float(pe_full[2]['ibf']['A']['mean_cp'] - pe_full[0]['ibf']['A']['mean_cp']),
    'BT_B': float(pe_full[2]['ibf']['B']['mean_cp'] - pe_full[1]['ibf']['B']['mean_cp']),
    'n_dissolved': sum(len(c.dissolution_log)
                       for c in ibf_full.value_centers + ibf_full.agency_centers),
}}
all_seeds.update(seed_results)

print(f"\n  {'Seed':<8}  {'vs Passive':>12}  {'BT_A':>8}  {'BT_B':>8}  {'Dissolved':>10}")
print("  " + "-" * 55)
for s in sorted(all_seeds):
    r = all_seeds[s]
    print(f"  {s:<8}  {r['vs_passive']:>+12.1f}  {r['BT_A']:>+8.1f}  "
          f"{r['BT_B']:>+8.1f}  {r['n_dissolved']:>10}")

vs_p = [all_seeds[s]['vs_passive'] for s in sorted(all_seeds)]
bt_a = [all_seeds[s]['BT_A'] for s in sorted(all_seeds)]
bt_b = [all_seeds[s]['BT_B'] for s in sorted(all_seeds)]
diss = [all_seeds[s]['n_dissolved'] for s in sorted(all_seeds)]

print(f"\n  {'':>8}  {'vs Passive':>12}  {'BT_A':>8}  {'BT_B':>8}  {'Dissolved':>10}")
print(f"  {'Mean':<8}  {np.mean(vs_p):>+12.1f}  {np.mean(bt_a):>+8.1f}  "
      f"{np.mean(bt_b):>+8.1f}  {np.mean(diss):>10.0f}")
print(f"  {'Std':<8}  {np.std(vs_p):>12.1f}  {np.std(bt_a):>8.1f}  "
      f"{np.std(bt_b):>8.1f}  {np.std(diss):>10.0f}")

# ── Save combined seed results ────────────────────────────────────────────────
out_seeds = {
    'seeds': {str(s): _js(all_seeds[s]) for s in sorted(all_seeds)},
    'summary': {
        'vs_passive_mean': float(np.mean(vs_p)),
        'vs_passive_std': float(np.std(vs_p)),
        'BT_A_mean': float(np.mean(bt_a)),
        'BT_A_std': float(np.std(bt_a)),
        'BT_B_mean': float(np.mean(bt_b)),
        'BT_B_std': float(np.std(bt_b)),
        'n_seeds': len(all_seeds),
    },
}
with open(os.path.join(CHECKPOINT_DIR, 'results_seeds.json'), 'w') as f:
    json_mod.dump(out_seeds, f, indent=2)
print(f"\n  Saved: results_seeds.json")
print(f"\n{'='*70}\n  C11 COMPLETE\n{'='*70}")


######################################################################
  SEED 43
######################################################################

  Full IBF Seed 43 | σ=1.2693, merge=1.9040, σ_agn=0.7806
  epochs=120, games=100, eval_n=500

──────────────────────────────────────────────────────────────────────
  PHASE A — Tactical/Imbalanced (epochs 1-120)
──────────────────────────────────────────────────────────────────────
  reset_verifications() called — all crystals must re-earn broadcast
  Ep   1 (A-  1): D=0.1720, val=234c/128x/0v, agn=55c/42x, sat=0%, 95.1s
    σ: mean=1.212, min=0.488, needles=2
  Ep  10 (A- 10): D=0.1653, val=693c/614x/0v, agn=127c/116x, sat=0%, 99.2s
    σ: mean=1.074, min=0.373, needles=20
  Ep  20 (A- 20): D=0.1676, val=952c/886x/0v, agn=165c/159x, sat=0%, 104.5s
    σ: mean=1.024, min=0.373, needles=31
  Ep  30 (A- 30): D=0.1711, val=1183c/1106x/0v, agn=195c/189x, sat=0%, 103.9s
    σ: mean=1.009, min=0.340, needles=46
  Ep  40 (A- 40): D=0.1742, 

In [16]:
# C12 — SEED EVALUATION AT PRESCRIBED σ_train
# ══════════════════════════════════════════════════════════════════════════════
# Evaluates seeds 43 and 44 at flat σ_train (prescribed geometry),
# matching the C9 evaluation done for seed 42.
#
# Seed 42 C9 result: +90.2 cp at flat σ_train = 1.2693
# Seeds 43, 44 C11 results: +60.3, +60.6 cp at trained geometry (with needles)
#
# This cell provides the three-seed headline at prescribed bandwidth.
# Requires: seed 43 and 44 checkpoints from C11.
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("  C12 — SEED EVALUATION AT PRESCRIBED σ_train")
print("=" * 70)

eval_all_c12 = []
for c in ['A', 'B', 'C']:
    eval_all_c12.extend(context_eval[c][:334])

p_load = build_params()

seed_eval_results = {}

# Seed 42: already have C9 result, but re-eval for consistency
seeds_to_eval = {
    42: os.path.join(CHECKPOINT_DIR, 'full_ibf_ph_C.pkl'),
    43: os.path.join(CHECKPOINT_DIR, 'full_ibf_seed_43.pkl'),
    44: os.path.join(CHECKPOINT_DIR, 'full_ibf_seed_44.pkl'),
}

sf_c12 = get_sf_engine(threads=2, hash_mb=256)

try:
    for seed_i, ckpt_path in seeds_to_eval.items():
        print(f"\n  Seed {seed_i}:")

        # Load engine
        if seed_i == 42 and 'ibf_full' in globals():
            eng_e = ibf_full
            print(f"    Using in-memory ibf_full")
        else:
            eng_e = load_engine(ckpt_path, p_load)

        # Capture trained sigmas
        trained_sigs = [c.sigma for c in eng_e.value_centers]
        n_needles = sum(1 for s in trained_sigs if s < SIGMA_TRAIN * 0.5)
        print(f"    Trained geometry: mean σ={np.mean(trained_sigs):.4f}, "
              f"min={np.min(trained_sigs):.4f}, needles={n_needles}")

        # Set flat σ_train
        for c in eng_e.value_centers:
            c.sigma = SIGMA_TRAIN
        print(f"    Evaluating at flat σ_train={SIGMA_TRAIN:.4f}...")

        # Eval IBF at flat σ_train
        sc_ibf = eval_agent_on_fens(eng_e, eval_all_c12, sf_c12)

        # Eval passive (same for all seeds)
        if seed_i == 42:
            sc_pas = eval_agent_on_fens(passive_agent, eval_all_c12, sf_c12)
            passive_scores_c12 = sc_pas
        else:
            sc_pas = passive_scores_c12

        nc_e = min(len(sc_ibf), len(sc_pas))
        d_e = sc_ibf[:nc_e] - sc_pas[:nc_e]
        try:    _, wp_e = scipy_stats.wilcoxon(d_e, zero_method='wilcox')
        except: wp_e = 1.0

        vs_pas = float(np.mean(d_e))
        ibf_mean = float(np.mean(sc_ibf[:nc_e]))
        n_clip = int(np.sum(np.abs(sc_ibf[:nc_e]) >= CP_CLIP_EVAL))

        print(f"    IBF at σ_train: {ibf_mean:+.1f} cp  "
              f"(vs passive: {vs_pas:+.1f}, p={wp_e:.4f}, clip={n_clip})")

        seed_eval_results[seed_i] = {
            'ibf_mean': ibf_mean,
            'vs_passive': vs_pas,
            'p_value': float(wp_e),
            'n_clip': n_clip,
            'per_position_ibf': sc_ibf[:nc_e].tolist(),
        }

        # Restore trained sigmas
        for c_obj, s in zip(eng_e.value_centers, trained_sigs):
            c_obj.sigma = s

        # Free loaded engines (keep ibf_full)
        if seed_i != 42:
            del eng_e
            gc.collect()

    # ── Cross-seed summary at prescribed σ_train ──────────────────────────────
    print(f"\n{'='*70}")
    print(f"  CROSS-SEED SUMMARY AT PRESCRIBED σ_train = {SIGMA_TRAIN:.4f}")
    print(f"{'='*70}")

    pas_mean = float(np.mean(passive_scores_c12))

    print(f"\n  {'Seed':<8}  {'IBF mean':>10}  {'vs Passive':>12}  {'p':>10}  {'Clip':>5}")
    print("  " + "-" * 52)
    for s in sorted(seed_eval_results):
        r = seed_eval_results[s]
        sig = '***' if r['p_value'] < 0.001 else ''
        print(f"  {s:<8}  {r['ibf_mean']:>+10.1f}  {r['vs_passive']:>+12.1f}  "
              f"{r['p_value']:.4f}{sig:>3}  {r['n_clip']:>5}")

    vs_p_all = [seed_eval_results[s]['vs_passive'] for s in sorted(seed_eval_results)]
    ibf_all  = [seed_eval_results[s]['ibf_mean'] for s in sorted(seed_eval_results)]

    print(f"\n  {'':>8}  {'IBF mean':>10}  {'vs Passive':>12}")
    print(f"  {'Mean':<8}  {np.mean(ibf_all):>+10.1f}  {np.mean(vs_p_all):>+12.1f}")
    print(f"  {'Std':<8}  {np.std(ibf_all):>10.1f}  {np.std(vs_p_all):>12.1f}")

    # ── Combined table: trained geometry vs prescribed σ ──────────────────────
    print(f"\n  --- Trained Geometry vs Prescribed σ_train ---")
    print(f"  {'Seed':<8}  {'Trained geom':>14}  {'Prescribed σ':>14}  {'Gain':>8}")
    print("  " + "-" * 48)

    trained_vs_pas = {42: 64.6, 43: 60.3, 44: 60.6}  # from C8/C11
    for s in sorted(seed_eval_results):
        t = trained_vs_pas[s]
        p = seed_eval_results[s]['vs_passive']
        print(f"  {s:<8}  {t:>+14.1f}  {p:>+14.1f}  {p-t:>+8.1f}")

    gains = [seed_eval_results[s]['vs_passive'] - trained_vs_pas[s]
             for s in sorted(seed_eval_results)]
    print(f"  {'Mean':>8}  {np.mean(list(trained_vs_pas.values())):>+14.1f}  "
          f"{np.mean(vs_p_all):>+14.1f}  {np.mean(gains):>+8.1f}")

    # ── Save ──────────────────────────────────────────────────────────────────
    out_c12 = {
        'sigma_train': SIGMA_TRAIN,
        'evaluation': 'flat_sigma_train',
        'seeds': {str(s): _js(seed_eval_results[s]) for s in sorted(seed_eval_results)},
        'summary': {
            'vs_passive_mean': float(np.mean(vs_p_all)),
            'vs_passive_std': float(np.std(vs_p_all)),
            'ibf_mean_mean': float(np.mean(ibf_all)),
            'ibf_mean_std': float(np.std(ibf_all)),
        },
        'passive_mean': pas_mean,
    }
    with open(os.path.join(CHECKPOINT_DIR, 'results_c12_prescribed_sigma.json'), 'w') as f:
        json_mod.dump(out_c12, f, indent=2)
    print(f"\n  Saved: results_c12_prescribed_sigma.json")

finally:
    sf_c12.quit()

print(f"\n{'='*70}\n  C12 COMPLETE\n{'='*70}")

  C12 — SEED EVALUATION AT PRESCRIBED σ_train

  Seed 42:
    Using in-memory ibf_full
    Trained geometry: mean σ=0.9375, min=0.2331, needles=284
    Evaluating at flat σ_train=1.2693...
    IBF at σ_train: -243.5 cp  (vs passive: +90.1, p=0.0000, clip=13)

  Seed 43:
  Loaded /workspace/ibf_chess_paper/full_ibf_seed_43.pkl: Val:6600c(6468x,4227v)|Agn:829c|Ep:360
    Trained geometry: mean σ=0.9337, min=0.1362, needles=321
    Evaluating at flat σ_train=1.2693...
    IBF at σ_train: -248.6 cp  (vs passive: +85.1, p=0.0000, clip=12)

  Seed 44:
  Loaded /workspace/ibf_chess_paper/full_ibf_seed_44.pkl: Val:6664c(6505x,4239v)|Agn:857c|Ep:360
    Trained geometry: mean σ=0.9316, min=0.2511, needles=352
    Evaluating at flat σ_train=1.2693...
    IBF at σ_train: -242.1 cp  (vs passive: +91.6, p=0.0000, clip=11)

  CROSS-SEED SUMMARY AT PRESCRIBED σ_train = 1.2693

  Seed        IBF mean    vs Passive           p   Clip
  ----------------------------------------------------
  42          

In [17]:
# C13 — THERMODYNAMIC SHRINK ABLATION
# ══════════════════════════════════════════════════════════════════════════════
# Surgical eval-time ablation isolating the contribution of thermodynamic
# shrink (ALPHA_SHRINK = 500) to the performance gap between trained
# geometry (+65.6 cp) and prescribed σ_train (+89.6 cp).
#
# Four evaluation configurations on the same trained engine (seed 42):
#   1. Trained geometry — all centers at their post-training σ
#   2. Needles widened — only σ < 0.5×σ_train reset to σ_train
#   3. All shrunk widened — all σ < 0.85×σ_train reset to σ_train
#   4. All flat σ_train — every center at prescribed σ_train
#
# Result: shrink accounts for 89% of the 23.4 cp gap. Needles (35%),
# general shrink (54%), noise (11%). ρ(σ,|v|) = -0.235 confirms shrink
# preferentially narrows the strongest corrections.
#
# Conclusion: thermodynamic shrink is a resolution artifact. The Crucible
# handles quarantine via dissolution. Shrink is redundant and harmful.
# Recommended: ALPHA_SHRINK = 0 in next iteration.
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("  C13 — THERMODYNAMIC SHRINK ABLATION")
print("=" * 70)

eval_all_c13 = []
for c in ['A', 'B', 'C']:
    eval_all_c13.extend(context_eval[c][:334])

# Capture trained sigmas
trained_sigs_c13 = [c.sigma for c in ibf_full.value_centers]

# Identify center populations
needle_threshold = SIGMA_TRAIN * 0.50
shrunk_threshold = SIGMA_TRAIN * 0.85

n_total    = len(trained_sigs_c13)
n_needles  = sum(1 for s in trained_sigs_c13 if s < needle_threshold)
n_shrunk   = sum(1 for s in trained_sigs_c13 if s < shrunk_threshold)
n_unshrunk = n_total - n_shrunk

print(f"\n  Center populations (n={n_total}):")
print(f"    Needles (σ < {needle_threshold:.3f}):     {n_needles}")
print(f"    Shrunk  (σ < {shrunk_threshold:.3f}):    {n_shrunk}")
print(f"    Unshrunk (σ ≥ {shrunk_threshold:.3f}):   {n_unshrunk}")
print(f"    σ range: [{np.min(trained_sigs_c13):.4f}, {np.max(trained_sigs_c13):.4f}]")
print(f"    σ mean:  {np.mean(trained_sigs_c13):.4f}")

# ── Per-center analysis: correction amplitude vs shrink status ────────────────
print(f"\n  ── Correction amplitude vs shrink status ────────────────────────")
v_needle  = [abs(c.v) for c, s in zip(ibf_full.value_centers, trained_sigs_c13)
             if s < needle_threshold]
v_shrunk  = [abs(c.v) for c, s in zip(ibf_full.value_centers, trained_sigs_c13)
             if needle_threshold <= s < shrunk_threshold]
v_normal  = [abs(c.v) for c, s in zip(ibf_full.value_centers, trained_sigs_c13)
             if s >= shrunk_threshold]

print(f"    Needles:  mean |v| = {np.mean(v_needle):.4f}  (n={len(v_needle)})")
print(f"    Shrunk:   mean |v| = {np.mean(v_shrunk):.4f}  (n={len(v_shrunk)})")
print(f"    Normal:   mean |v| = {np.mean(v_normal):.4f}  (n={len(v_normal)})")

# Correlation between σ and |v|
all_sigs = np.array(trained_sigs_c13)
all_abs_v = np.array([abs(c.v) for c in ibf_full.value_centers])
rho_sv, p_sv = scipy_stats.spearmanr(all_sigs, all_abs_v)
print(f"    ρ(σ, |v|) = {rho_sv:.3f}, p = {p_sv:.4f}")
if abs(rho_sv) < 0.1:
    print(f"    ✓ Correction amplitudes uncorrelated with shrink — "
          f"shrink affects bandwidth, not learned content")
else:
    print(f"    Note: weak correlation detected — shrink may interact with learning")

# ── Surgical evaluation ───────────────────────────────────────────────────────
print(f"\n  ── Surgical evaluation (n={len(eval_all_c13)}) ─────────────────")

sf_c13 = get_sf_engine(threads=2, hash_mb=256)

configurations = []

try:
    # Config 1: Trained geometry (needles intact)
    for c, s in zip(ibf_full.value_centers, trained_sigs_c13):
        c.sigma = s
    sc_trained = eval_agent_on_fens(ibf_full, eval_all_c13, sf_c13)

    # Config 2: Only needles widened to σ_train
    for c, s in zip(ibf_full.value_centers, trained_sigs_c13):
        c.sigma = SIGMA_TRAIN if s < needle_threshold else s
    sc_needles_widened = eval_agent_on_fens(ibf_full, eval_all_c13, sf_c13)

    # Config 3: All shrunk centers widened to σ_train
    for c, s in zip(ibf_full.value_centers, trained_sigs_c13):
        c.sigma = SIGMA_TRAIN if s < shrunk_threshold else s
    sc_shrunk_widened = eval_agent_on_fens(ibf_full, eval_all_c13, sf_c13)

    # Config 4: All flat σ_train
    for c in ibf_full.value_centers:
        c.sigma = SIGMA_TRAIN
    sc_flat = eval_agent_on_fens(ibf_full, eval_all_c13, sf_c13)

    # Passive baseline
    sc_pas = eval_agent_on_fens(passive_agent, eval_all_c13, sf_c13)

    # Compute vs passive for each
    nc13 = min(len(sc_trained), len(sc_needles_widened),
               len(sc_shrunk_widened), len(sc_flat), len(sc_pas))
    pas = sc_pas[:nc13]

    configs = [
        ('Trained geometry',     sc_trained[:nc13]),
        ('Needles widened',      sc_needles_widened[:nc13]),
        ('All shrunk widened',   sc_shrunk_widened[:nc13]),
        ('All flat σ_train',     sc_flat[:nc13]),
    ]

    print(f"\n  {'Configuration':<25}  {'Mean cp':>8}  {'vs Passive':>12}  {'W-p':>10}")
    print("  " + "-" * 60)
    for name, sc in configs:
        d = sc - pas
        try:    _, wp = scipy_stats.wilcoxon(d, zero_method='wilcox')
        except: wp = 1.0
        sig = '***' if wp < 0.001 else ''
        print(f"  {name:<25}  {np.mean(sc):>+8.1f}  {np.mean(d):>+12.1f}  "
              f"{wp:.4f}{sig:>3}")

    # MLP and Replay reference
    print(f"\n  Reference baselines:")
    print(f"    MLP:    +82.3 cp vs passive")
    print(f"    Replay: +76.4 cp vs passive")

    # ── Pairwise comparisons ──────────────────────────────────────────────────
    print(f"\n  ── Pairwise attribution ──────────────────────────────────────")
    vs_p = {name: float(np.mean(sc - pas)) for name, sc in configs}

    total_gap = vs_p['All flat σ_train'] - vs_p['Trained geometry']
    needle_contrib = vs_p['Needles widened'] - vs_p['Trained geometry']
    shrunk_contrib = vs_p['All shrunk widened'] - vs_p['Needles widened']
    normal_contrib = vs_p['All flat σ_train'] - vs_p['All shrunk widened']

    print(f"    Total gap (trained → flat):     {total_gap:+.1f} cp")
    print(f"    From widening needles:          {needle_contrib:+.1f} cp  "
          f"({100*needle_contrib/total_gap:.0f}% of gap)")
    print(f"    From widening other shrunk:     {shrunk_contrib:+.1f} cp  "
          f"({100*shrunk_contrib/total_gap:.0f}% of gap)")
    print(f"    From widening normal centers:   {normal_contrib:+.1f} cp  "
          f"({100*normal_contrib/total_gap:.0f}% of gap)")

finally:
    sf_c13.quit()

# Restore trained geometry
for c, s in zip(ibf_full.value_centers, trained_sigs_c13):
    c.sigma = s

# ── Save ──────────────────────────────────────────────────────────────────────
out_c13 = {
    'sigma_train': SIGMA_TRAIN,
    'populations': {
        'total': n_total,
        'needles': n_needles,
        'shrunk': n_shrunk,
        'unshrunk': n_unshrunk,
    },
    'correction_amplitude': {
        'needle_mean_abs_v': float(np.mean(v_needle)),
        'shrunk_mean_abs_v': float(np.mean(v_shrunk)),
        'normal_mean_abs_v': float(np.mean(v_normal)),
        'rho_sigma_v': float(rho_sv),
        'rho_sigma_v_p': float(p_sv),
    },
    'configurations': {name: float(np.mean(sc - pas)) for name, sc in configs},
    'attribution': {
        'total_gap': total_gap,
        'needle_contribution': needle_contrib,
        'shrunk_contribution': shrunk_contrib,
        'normal_contribution': normal_contrib,
    },
}
with open(os.path.join(CHECKPOINT_DIR, 'results_c13_shrink_ablation.json'), 'w') as f:
    json_mod.dump(_js(out_c13), f, indent=2)
print(f"\n  Saved: results_c13_shrink_ablation.json")
print(f"\n{'='*70}\n  C13 COMPLETE\n{'='*70}")

  C13 — THERMODYNAMIC SHRINK ABLATION

  Center populations (n=6680):
    Needles (σ < 0.635):     284
    Shrunk  (σ < 1.079):    5167
    Unshrunk (σ ≥ 1.079):   1513
    σ range: [0.2331, 1.2693]
    σ mean:  0.9375

  ── Correction amplitude vs shrink status ────────────────────────
    Needles:  mean |v| = 0.0462  (n=284)
    Shrunk:   mean |v| = 0.0400  (n=4883)
    Normal:   mean |v| = 0.0268  (n=1513)
    ρ(σ, |v|) = -0.235, p = 0.0000
    Note: weak correlation detected — shrink may interact with learning

  ── Surgical evaluation (n=1002) ─────────────────

  Configuration               Mean cp    vs Passive         W-p
  ------------------------------------------------------------
  Trained geometry             -266.2         +66.2  0.0000***
  Needles widened              -258.0         +74.4  0.0000***
  All shrunk widened           -245.4         +87.0  0.0000***
  All flat σ_train             -242.8         +89.6  0.0000***

  Reference baselines:
    MLP:    +82.3 cp vs

## Note: Thermodynamic Shrink & "Train Tight, Evaluate Wide"

Some code comments and print statements in C9 and C10 reference "train tight, evaluate wide" and thermodynamic shrink as if they are active features. Based on the C13 ablation and C12 three-seed evaluation, both are retired from the paper narrative:

- **Thermodynamic shrink** (ALPHA_SHRINK = 500) compresses 77% of centers below prescribed σ,
  accounting for 89% of the performance gap between trained geometry and prescribed σ (C13).
  The Crucible handles quarantine via dissolution. Shrink is redundant and harmful.
- **"Train tight, evaluate wide"** was an artifact of shrink, not a finding. At prescribed σ,
  all three domains peak within ±10% of σ*. No two-bandwidth story is needed.

**Paper numbers use prescribed σ_train evaluation (C9/C12): +88.9 ± 2.8 cp (3 seeds).**

Next iteration: set ALPHA_SHRINK = 0, remove shrink equation from engine, remove σ_floor
derivation, clean stale comments in C9/C10. Until then, the code runs correctly, the stale references are cosmetic and do not affect any computed result or saved data.